# IMF Article IV PDF → Dashboard: an LLM extraction, evaluation and RAG pipeline

**Report analysed:** *Dominica: 2024 Article IV Consultation – Press Release; Staff Report; and Statement by the Executive Director for Dominica* (IMF Country Report No. 24/192, June 2024).
**Allocation:** SRN 24160777 → Dominica, 2023–2024.

This notebook runs end-to-end and builds:

| Task | What the notebook does | Main outputs |
|---|---|---|
| 1 | PDF ingestion (`pypdf`), cleaning, section detection, token-aware chunking, section summaries (<100 words), executive summary, **dual-LLM judge + refinement loop** | `outputs/sections.json`, `outputs/summaries.json`, `outputs/executive_summary.json` |
| 2 | 7-theme distribution (lexicon + embedding classifier), strengths/vulnerabilities, **schema-constrained JSON indicators** with fallbacks, temporal trends, judge audit | `outputs/themes.json`, `outputs/risk_insights.json`, `outputs/core_indicators.json`, `outputs/indicator_audit.json`, `outputs/temporal_trends.json` |
| 3 | Local RAG: `sentence-transformers` embeddings + FAISS `IndexFlatL2`, grounded Q&A, closed-book hallucination baseline | `outputs/rag_results.json` |
| 4 | `pd.melt` tidy transforms + a 6-panel, 300-dpi dashboard | `figures/dashboard.png` + individual panels |
| Extension | **(a)** model and system-instruction benchmarking, **(b)** Multi-Query RAG vs single-query RAG | `outputs/benchmark.json`, `outputs/rag_results.json` |

### How to run
1. **Runtime → Run all.** In Colab, add your Gemini key under **🔑 Secrets** as `GEMINI_API_KEY` and enable notebook access.
2. **No key? It still runs.** Every LLM response is cached on disk (`cache/llm/`). When the cache is present (it is committed to the GitHub repo), the notebook replays the cached responses with no API key and no network calls. Set `CONFIG["repo_url"]` so that Colab clones the cache automatically.
3. The PDF is downloaded from imf.org. If that fails, Colab asks you to upload it.

> Design principle: the LLM is never trusted blindly. Every generated artefact is (i) schema-validated, (ii) audited by a *second* LLM acting as judge, and (iii) where possible checked deterministically (quote verification, table-parser cross-checks, human ground truth).

## 0. Setup
Installs the packages Colab doesn't ship with. `pypdf` handles parsing, `google-genai` is the Gemini SDK, `sentence-transformers` + `faiss-cpu` power local RAG, and `tiktoken` is the fallback tokenizer.

In [ ]:
%pip install -q "pypdf>=4.2" "google-genai>=1.20" "sentence-transformers>=3.0" "faiss-cpu>=1.8" "tiktoken>=0.7"

In [ ]:
# ---- Standard library ------------------------------------------------------
import os, re, sys, json, time, math, hashlib, unicodedata, warnings, subprocess, textwrap
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any, Tuple
from collections import Counter, defaultdict

# ---- Third party -----------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pydantic import BaseModel, Field, ValidationError
from pypdf import PdfReader
import faiss
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types as genai_types
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
IN_COLAB = "google.colab" in sys.modules
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | running in Colab: {IN_COLAB}")

### 0.1 Configuration
All tunable parameters live in one dictionary, so the notebook body has no magic numbers. The chunk sizes follow the brief (300–500 tokens, 50-token overlap). The embedding model is `BAAI/bge-small-en-v1.5`, which has a **512-token** input window, so a 400-token chunk is embedded whole. (`all-MiniLM-L6-v2` would silently truncate anything past 256 tokens.)

Model names are *preferences*. The notebook asks the Gemini API which models are actually available and picks the first match. The resolved names are saved to `outputs/run_config.json` so that cached replays use exactly the same models.

In [ ]:
CONFIG: Dict[str, Any] = {
    # --- Assignment allocation ---------------------------------------------
    "student_id": "24160777",
    "country": "Dominica",
    "confusable_country": "the Dominican Republic",   # explicit guard used in prompts
    "report_period": "2023-2024",
    "report_title": ("Dominica: 2024 Article IV Consultation-Press Release; Staff Report; "
                     "and Statement by the Executive Director for Dominica"),
    "imf_country_report_no": "IMF Country Report No. 24/192",
    "pdf_url": "https://www.imf.org/-/media/files/publications/cr/2024/english/1dmaea2024001-print-pdf.pdf",
    "pdf_filename": "dominica_2024_article_iv.pdf",
    "repo_url": "",                 # e.g. "https://github.com/<user>/imf-llm-pipeline" (enables key-less replay in Colab)

    # --- Chunking -----------------------------------------------------------
    "chunk_target_tokens": 400,
    "chunk_overlap_tokens": 50,
    "chunk_min_tokens": 60,
    "chunk_hard_max_tokens": 500,   # below the embedder's 512-token window

    # --- Retrieval ------------------------------------------------------------
    "embedding_model": "BAAI/bge-small-en-v1.5",
    "embedding_query_prefix": "Represent this sentence for searching relevant passages: ",
    "retrieval_k": 5,
    "multi_query_n_variants": 3,
    "multi_query_top_k": 8,
    "rrf_k": 60,

    # --- Summarisation / judging -------------------------------------------
    "summary_word_limit": 100,      # brief: "fewer than 100 words"
    "summary_target_words": 85,     # prompt target leaves a safety margin
    "section_max_tokens": 12_000,   # longer sections are summarised map-reduce style
    "exec_summary_max_input_tokens": 120_000,
    "judge_pass_threshold": 4,      # every criterion must be >= 4 out of 5
    "max_refinement_rounds": 1,

    # --- Time series --------------------------------------------------------
    "projection_start_year": 2024,  # Table 1 marks 2023 as "Est." and 2024+ as "Projected"

    # --- LLM ----------------------------------------------------------------
    "temperature": None,            # None = model default (recommended for Gemini 3.x); reproducibility comes from the cache
    "min_seconds_between_calls": 4.0,   # free-tier friendly throttle
    "max_retries": 6,
    "extractor_model_preferences": ["gemini-3.5-flash", "gemini-3-flash", "gemini-2.5-flash", "gemini-2.0-flash"],
    "judge_model_preferences":     ["gemini-3.8-flash", "gemini-3.5-pro", "gemini-3-pro", "gemini-2.5-pro",
                                    "gemini-3-flash", "gemini-2.5-flash"],
    "lite_model_preferences":      ["gemini-3.5-flash-lite", "gemini-3-flash-lite", "gemini-2.5-flash-lite",
                                    "gemini-2.0-flash-lite"],
}

THEMES = ["Inflation & Prices", "Interest Rates & Credit", "Housing & Real Estate",
          "Banking Capital & Liquidity", "Sovereign Debt", "Labor Market Dynamics",
          "Systemic Financial Risk"]


def resolve_base_dir() -> Path:
    """Work in the repo folder if we are in one; in Colab, optionally clone the repo to get the cache."""
    env = os.environ.get("IMF_PIPELINE_DIR")
    if env:
        return Path(env).resolve()
    if IN_COLAB and CONFIG["repo_url"]:
        target = Path("/content") / Path(CONFIG["repo_url"].rstrip("/")).stem
        if not target.exists():
            subprocess.run(["git", "clone", "--depth", "1", CONFIG["repo_url"], str(target)], check=False)
        if target.exists():
            return target
    return Path.cwd()


BASE_DIR = resolve_base_dir()
DATA_DIR, CACHE_DIR = BASE_DIR / "data", BASE_DIR / "cache"
OUT_DIR, FIG_DIR = BASE_DIR / "outputs", BASE_DIR / "figures"
for d in (DATA_DIR, CACHE_DIR / "llm", CACHE_DIR / "embeddings", OUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
PDF_PATH = DATA_DIR / CONFIG["pdf_filename"]
print("Working directory:", BASE_DIR)


def save_json(obj: Any, name: str) -> Path:
    """Write a machine-readable artefact to outputs/ (numpy types are converted)."""
    def _default(o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        if isinstance(o, BaseModel): return o.model_dump()
        if isinstance(o, Path): return str(o)
        raise TypeError(f"Not JSON serialisable: {type(o)}")
    path = OUT_DIR / name
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=_default), encoding="utf-8")
    return path

### 0.2 LLM client: disk cache, throttling, retries, schema parsing
One wrapper handles every Gemini call. It:
* **Caches** each response on disk under a SHA-256 key of (model, system instruction, prompt, schema, temperature). Re-runs cost no quota and replay deterministically, cached latency included, so benchmark numbers stay stable.
* **Throttles** calls to respect free-tier rate limits, and **retries** transient errors (429/5xx) with exponential backoff.
* **Enforces schemas**. When a Pydantic model is passed, Gemini runs in JSON mode with `response_schema`, and the reply is validated again locally. Validation failures are *recorded* rather than raised, so that schema compliance can be measured.

The API key comes from Colab Secrets or the environment. It is never printed or written to disk.

In [ ]:
def get_api_key() -> Optional[str]:
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not key and IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None
    return key or None


@dataclass
class LLMResult:
    text: str
    parsed: Any
    model: str
    latency_s: float
    cached: bool
    tag: str = ""
    error: Optional[str] = None

    @property
    def ok(self) -> bool:
        return self.error is None


def extract_json_block(text: str) -> Optional[Any]:
    """Robustly parse JSON from a model reply (handles ```json fences and leading prose)."""
    if not text:
        return None
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.MULTILINE)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{.*\}|\[.*\]", cleaned, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            return None
    return None


class LLMClient:
    TRANSIENT = ("429", "500", "502", "503", "504", "RESOURCE_EXHAUSTED", "UNAVAILABLE",
                 "DEADLINE", "overloaded", "timed out", "Timeout")

    def __init__(self, api_key: Optional[str], cache_dir: Path,
                 min_interval: float = 4.0, max_retries: int = 6):
        self.client = genai.Client(api_key=api_key) if api_key else None
        self.cache_dir = cache_dir
        self.min_interval = min_interval
        self.max_retries = max_retries
        self._last_call = 0.0
        self.log: List[Dict[str, Any]] = []

    # -- low-level API call (isolated so it can be swapped/mocked) -----------
    def _call_api(self, model: str, prompt: str, system: Optional[str],
                  schema: Optional[type], temperature: Optional[float]) -> str:
        cfg: Dict[str, Any] = {}
        if system:
            cfg["system_instruction"] = system
        if schema is not None:
            cfg["response_mime_type"] = "application/json"
            cfg["response_schema"] = schema
        if temperature is not None:
            cfg["temperature"] = temperature
        resp = self.client.models.generate_content(
            model=model, contents=prompt, config=genai_types.GenerateContentConfig(**cfg))
        return resp.text or ""

    def _cache_key(self, model, prompt, system, schema, temperature) -> str:
        payload = {"model": model, "system": system, "prompt": prompt, "temperature": temperature,
                   "schema": schema.model_json_schema() if schema is not None else None}
        return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()

    def generate(self, prompt: str, model: str, system: Optional[str] = None,
                 schema: Optional[type] = None, temperature: Optional[float] = CONFIG["temperature"],
                 tag: str = "") -> LLMResult:
        key = self._cache_key(model, prompt, system, schema, temperature)
        path = self.cache_dir / f"{key}.json"
        text, latency, cached, error = "", 0.0, False, None

        if path.exists():
            rec = json.loads(path.read_text(encoding="utf-8"))
            text, latency, cached = rec["text"], rec["latency_s"], True
        elif self.client is None:
            error = ("No API key available and this prompt is not in the cache. "
                     "Set GEMINI_API_KEY (Colab Secrets) or clone the repo that contains cache/llm/.")
        else:
            for attempt in range(self.max_retries):
                wait = self.min_interval - (time.time() - self._last_call)
                if wait > 0:
                    time.sleep(wait)
                try:
                    t0 = time.perf_counter()
                    self._last_call = time.time()
                    text = self._call_api(model, prompt, system, schema, temperature)
                    latency = time.perf_counter() - t0
                    error = None
                    break
                except Exception as exc:                     # noqa: BLE001 - we classify below
                    error = f"{type(exc).__name__}: {str(exc)[:300]}"
                    if not any(t in str(exc) for t in self.TRANSIENT):
                        break                                 # permanent error: don't burn retries
                    time.sleep(min(60, 5 * 2 ** attempt))
            if error is None:
                path.write_text(json.dumps({"tag": tag, "model": model, "latency_s": latency,
                                            "created": time.strftime("%Y-%m-%d %H:%M:%S"),
                                            "prompt_preview": prompt[:300], "text": text},
                                           ensure_ascii=False, indent=1), encoding="utf-8")

        parsed = None
        if schema is not None and error is None:
            raw = extract_json_block(text)
            try:
                parsed = schema.model_validate(raw)
            except (ValidationError, TypeError) as exc:
                error = f"SchemaValidationError: {str(exc)[:300]}"
        self.log.append({"tag": tag, "model": model, "latency_s": round(latency, 3), "cached": cached,
                         "ok": error is None, "error": error})
        return LLMResult(text=text, parsed=parsed, model=model, latency_s=latency,
                         cached=cached, tag=tag, error=error)


API_KEY = get_api_key()
llm = LLMClient(API_KEY, CACHE_DIR / "llm", CONFIG["min_seconds_between_calls"], CONFIG["max_retries"])
print("Gemini API key found:", API_KEY is not None,
      "| cached responses:", len(list((CACHE_DIR / "llm").glob("*.json"))))

### 0.3 Model resolution (dual-LLM set-up)
* **Extractor/summariser:** a Gemini *Flash* model, chosen for speed and cost.
* **Judge:** a *different* model, preferably a stronger or newer one. A model grading its own output shows **self-preference bias** (Zheng et al., 2023), so the judge is always a different model whenever one is available.
* **Lite:** a Flash-Lite variant, used only in the benchmarking extension.

In [ ]:
def list_available_models(client) -> List[str]:
    names = []
    for m in client.models.list():
        actions = getattr(m, "supported_actions", None) or []
        if actions and "generateContent" not in actions:
            continue
        names.append(m.name.split("/")[-1])
    return names


def pick_model(prefs: List[str], available: List[str], exclude: Tuple[str, ...] = ()) -> Optional[str]:
    """Exact match first, then the first available name that starts with the preference
    (e.g. 'gemini-2.5-flash' matches 'gemini-2.5-flash-001'), never mixing in lite or special variants."""
    banned = ("image", "tts", "audio", "live", "embedding", "vision")
    for p in prefs:
        if p in available and p not in exclude:
            return p
        for a in sorted(available, reverse=True):
            if a.startswith(p) and a not in exclude and not any(b in a for b in banned) \
                    and ("lite" in p or "lite" not in a):
                return a
    return None


RUN_CONFIG_PATH = OUT_DIR / "run_config.json"
if llm.client is not None:
    try:
        AVAILABLE = list_available_models(llm.client)
    except Exception as exc:                                  # noqa: BLE001
        print("Could not list models (", exc, ") – using first preferences.")
        AVAILABLE = []
    if AVAILABLE:
        EXTRACTOR_MODEL = pick_model(CONFIG["extractor_model_preferences"], AVAILABLE)
        JUDGE_MODEL = pick_model(CONFIG["judge_model_preferences"], AVAILABLE, exclude=(EXTRACTOR_MODEL,)) \
            or EXTRACTOR_MODEL
        LITE_MODEL = pick_model(CONFIG["lite_model_preferences"], AVAILABLE)
    else:
        EXTRACTOR_MODEL = CONFIG["extractor_model_preferences"][0]
        JUDGE_MODEL = CONFIG["judge_model_preferences"][0]
        LITE_MODEL = CONFIG["lite_model_preferences"][0]
    save_json({"extractor_model": EXTRACTOR_MODEL, "judge_model": JUDGE_MODEL, "lite_model": LITE_MODEL,
               "config": CONFIG, "resolved_at": time.strftime("%Y-%m-%d %H:%M:%S")}, "run_config.json")
elif RUN_CONFIG_PATH.exists():                                # key-less replay from cache
    _rc = json.loads(RUN_CONFIG_PATH.read_text())
    EXTRACTOR_MODEL, JUDGE_MODEL, LITE_MODEL = _rc["extractor_model"], _rc["judge_model"], _rc["lite_model"]
else:
    raise RuntimeError("No GEMINI_API_KEY and no outputs/run_config.json cache – cannot run the LLM steps.")

if EXTRACTOR_MODEL is None:
    raise RuntimeError("No Gemini Flash model is available for this key. Edit CONFIG['extractor_model_preferences'].")
print(f"Extractor: {EXTRACTOR_MODEL} | Judge: {JUDGE_MODEL} | Lite (benchmark): {LITE_MODEL}")
if JUDGE_MODEL == EXTRACTOR_MODEL:
    print("⚠️  Judge is the same model as the extractor – self-preference bias is likely; discuss in the report.")

---
## Task 1 — PDF processing pipeline and summarisation
### 1.1 Document ingestion
The PDF is downloaded from imf.org (with an upload fallback in Colab). `pypdf` extracts every page **twice**:
* **`plain` mode** follows the PDF's internal content-stream order. That is good for running prose.
* **`layout` mode** rebuilds the *visual* positions of the text. It is essential for tables. On p.36 of this report, Tables 5a (Commercial Banks) and 5b (Credit Unions) share a page, and the content stream emits the credit-union block *first*. Plain-mode parsing therefore pins credit-union NPLs on banks. Layout mode keeps each block under its correct title.

Extracted pages are cached to `data/pages.json`, keyed by the PDF's SHA-256.

In [ ]:
def ensure_pdf(path: Path, url: str) -> Path:
    """Return a local copy of the IMF report: existing file → download → Colab upload."""
    if path.exists() and path.stat().st_size > 10_000:
        return path
    try:
        import urllib.request
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (academic-research-notebook)"})
        with urllib.request.urlopen(req, timeout=90) as resp:
            data = resp.read()
        if data[:4] == b"%PDF":
            path.write_bytes(data)
            print(f"Downloaded {len(data)/1e6:.1f} MB from imf.org")
            return path
        print("Download did not return a PDF (got", data[:20], ")")
    except Exception as exc:                                  # noqa: BLE001
        print("Automatic download failed:", exc)
    if IN_COLAB:
        from google.colab import files
        print(f"Please upload the IMF PDF ({CONFIG['imf_country_report_no']}).")
        uploaded = files.upload()
        name, content = next(iter(uploaded.items()))
        path.write_bytes(content)
        return path
    raise FileNotFoundError(f"Place the report at {path} (source: {url})")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def extract_pages(pdf_path: Path) -> List[Dict[str, Any]]:
    """Extract plain + layout text per page, caching the result keyed by the PDF hash."""
    digest = sha256_file(pdf_path)
    cache = DATA_DIR / "pages.json"
    if cache.exists():
        rec = json.loads(cache.read_text(encoding="utf-8"))
        if rec.get("sha256") == digest:
            return rec["pages"]
    reader = PdfReader(str(pdf_path))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        plain = page.extract_text() or ""
        try:
            layout = page.extract_text(extraction_mode="layout") or plain
        except Exception:                                     # noqa: BLE001 - very old pypdf / odd page
            layout = plain
        pages.append({"page": i, "plain": plain, "layout": layout})
    cache.write_text(json.dumps({"sha256": digest, "pages": pages}, ensure_ascii=False), encoding="utf-8")
    return pages


PDF_PATH = ensure_pdf(PDF_PATH, CONFIG["pdf_url"])
RAW_PAGES = extract_pages(PDF_PATH)
print(f"{len(RAW_PAGES)} pages | {sum(len(p['plain']) for p in RAW_PAGES):,} characters (plain mode)")
print("--- first 400 characters of page 2 (raw) ---\n", RAW_PAGES[1]["plain"][:400])

### 1.2 Cleaning: unicode, running headers/footers, artefacts
IMF reports produce a few typical kinds of noise, each handled by a separate function:

| Noise | Example from this report | Fix |
|---|---|---|
| Unicode fractions | `43½ percentage point`, `9¾ percent` | → `43.5`, `9.75` (otherwise the LLM sees "43" and "½") |
| Ligatures, typographic dashes and quotes | `ﬁ`, `–`, `’` | NFKC normalisation + ASCII mapping |
| Running headers/footers | `DOMINICA`, `INTERNATIONAL MONETARY FUND 13` | **learned** from line frequency at page edges, not hard-coded |
| Table-of-contents leaders | `CONTEXT_____ 4` | dropped |
| Hard line wraps | sentences split mid-line | paragraph reflow (numbered IMF paragraphs start new blocks) |

Lines with three or more numbers (table rows, year headers) are never treated as boilerplate, so the table parser can still see them.

In [ ]:
FRACTIONS = {"½": ".5", "¼": ".25", "¾": ".75", "⅓": ".33", "⅔": ".67", "⅛": ".125"}
CHAR_MAP = {"‘": "'", "’": "'", "“": '"', "”": '"', "–": "-", "—": "-",
            "−": "-", "‐": "-", "‑": "-", " ": " ", "…": "...", "•": "•",
            "­": ""}
NUM_TOKEN = re.compile(r"^\(?-?\d{1,3}(?:,\d{3})+(?:\.\d+)?\)?%?$|^\(?-?\d+(?:\.\d+)?\)?%?$")
MISSING_TOKEN = re.compile(r"^(?:n\.?a\.?|\.\.\.|…|-{1,2}|—)$", re.IGNORECASE)


def normalise_text(text: str) -> str:
    for frac, dec in FRACTIONS.items():              # must run BEFORE NFKC (NFKC turns ½ into 1⁄2)
        text = re.sub(rf"(\d){frac}", lambda m, d=dec: m.group(1) + d, text)
        text = text.replace(frac, "0" + dec)
    text = unicodedata.normalize("NFKC", text)
    for src, dst in CHAR_MAP.items():
        text = text.replace(src, dst)
    return text


def n_numbers(line: str) -> int:
    return sum(bool(NUM_TOKEN.match(t)) for t in line.split())


def is_table_row(line: str) -> bool:
    toks = line.split()
    nums = sum(bool(NUM_TOKEN.match(t) or MISSING_TOKEN.match(t)) for t in toks)
    return nums >= 4 and nums / max(1, len(toks)) >= 0.4


def boilerplate_key(line: str) -> str:
    return re.sub(r"\d+", "#", re.sub(r"\s+", " ", line.strip()))


def learn_boilerplate(pages_lines: List[List[str]], edge: int = 2, min_share: float = 0.2) -> set:
    """Lines that recur in the first/last `edge` lines of >= `min_share` of pages are running headers/footers."""
    counts = Counter()
    for lines in pages_lines:
        nonempty = [l for l in lines if l.strip()]
        for l in set(nonempty[:edge] + nonempty[-edge:]):
            ls = l.strip()
            if n_numbers(ls) < 3 and len(ls) < 80 and len(ls.split()) <= 8 and not ls.endswith("."):
                counts[boilerplate_key(l)] += 1
    n = max(1, len(pages_lines))
    return {k for k, v in counts.items() if v / n >= min_share}


ARTEFACT_PATTERNS = [
    re.compile(r"^\s*\d{1,3}\s*$"),                       # bare page numbers
    re.compile(r"_{4,}"),                                 # TOC leader lines
    re.compile(r"^\s*©\s*\d{4}\s+International Monetary Fund", re.I),
]


def clean_page_lines(text: str, boilerplate: set) -> List[str]:
    out = []
    for line in normalise_text(text).splitlines():
        s = re.sub(r"[ \t]+", " ", line).strip()
        if not s or boilerplate_key(s) in boilerplate or any(p.search(s) for p in ARTEFACT_PATTERNS):
            continue
        out.append(s)
    return out


def clean_layout_lines(text: str, boilerplate: set) -> List[str]:
    """Layout text keeps its column spacing (collapsed to max 3 spaces) so table rows stay intact."""
    out = []
    for line in normalise_text(text).splitlines():
        s = re.sub(r" {3,}", "   ", line.rstrip())
        if not s.strip() or boilerplate_key(s) in boilerplate or any(p.search(s) for p in ARTEFACT_PATTERNS):
            continue
        out.append(s.strip())
    return out


_plain_lines = [normalise_text(p["plain"]).splitlines() for p in RAW_PAGES]
_layout_lines = [[re.sub(r"\s+", " ", l) for l in normalise_text(p["layout"]).splitlines()] for p in RAW_PAGES]
BOILERPLATE = learn_boilerplate(_plain_lines) | learn_boilerplate(_layout_lines)   # layout mode merges header lines
PAGES = [{"page": p["page"],
          "lines": clean_page_lines(p["plain"], BOILERPLATE),
          "layout_lines": clean_layout_lines(p["layout"], BOILERPLATE)} for p in RAW_PAGES]

print("Learned running headers/footers:", sorted(BOILERPLATE))
raw_chars = sum(len(p["plain"]) for p in RAW_PAGES)
clean_chars = sum(len(" ".join(p["lines"])) for p in PAGES)
print(f"Characters: raw {raw_chars:,} → cleaned {clean_chars:,} ({1 - clean_chars / raw_chars:.1%} removed)")
print("--- page 9 after cleaning (first 8 lines) ---")
print("\n".join(PAGES[min(8, len(PAGES) - 1)]["lines"][:8]))

### 1.3 Document structure: parts and sections
An Article IV *package* bundles several documents. Parts are detected from markers near the top of each page and are only allowed to move **forward** (each later cover page also repeats "STAFF REPORT FOR THE 2024 ARTICLE IV CONSULTATION", so a naive search would jump back):

`Press Release → Staff Report → Informational Annex → Debt Sustainability Analysis → Executive Director Statement`

Inside the Staff Report, chapters are detected from layout cues instead of a hard-coded list: stand-alone ALL-CAPS headings (`RECENT DEVELOPMENTS`), `Annex I.` headings, and the start of the figures/tables block. The shorter parts are summarised as single units.

In [ ]:
PART_MARKERS = [  # (part name, regex searched in the first ~600 characters of a page) – checked latest-first
    ("Executive Director Statement", re.compile(r"(?m)^Statement by ")),
    ("Debt Sustainability Analysis", re.compile(r"DEBT SUSTAINABILITY\s+ANALYSIS")),
    ("Informational Annex", re.compile(r"INFORMATIONAL\s+ANNEX")),
    ("Staff Report", re.compile(r"STAFF REPORT FOR THE \d{4}\s+ARTICLE IV")),
    ("Press Release", re.compile(r"FOR IMMEDIATE RELEASE|^PR\s?\d{2}/\d+", re.M)),
]
PART_ORDER = ["Cover", "Press Release", "Staff Report", "Informational Annex",
              "Debt Sustainability Analysis", "Executive Director Statement"]
WHOLE_PARTS = {"Press Release", "Informational Annex", "Debt Sustainability Analysis",
               "Executive Director Statement"}
HEADING_STOPLIST = {"CONTENTS", "FIGURES", "TABLES", "BOXES", "ANNEXES", "APPENDICES", "APPENDIX",
                    "FOR IMMEDIATE RELEASE", "APPROVED BY", "ANNEX", "SOURCES", "NOTES"}
UPPER_HEADING = re.compile(r"^[A-Z][A-Z0-9 ,&'\-:()/]{3,90}$")
ANNEX_HEADING = re.compile(r"^Annex\s+([IVXL]+)\.\s+(?!(?:Figure|Table|Box)\b)(\S.+)$")
FIG_TAB_START = re.compile(r"^(?:Figure|Table)\s+1[a-z]?\.\s+[A-Z][\w' ]+:")
TITLE_LINE = re.compile(r"^(?:Annex\s+[IVXL]+\.\s+)?(?:Table|Figure|Box)\s+\d+[a-z]?\.\s+\S")


def detect_part(page_text_head: str, current: str) -> str:
    for name, rx in PART_MARKERS:
        if rx.search(page_text_head) and PART_ORDER.index(name) > PART_ORDER.index(current):
            return name
    return current


def is_upper_heading(line: str) -> bool:
    words = line.split()
    if not UPPER_HEADING.match(line) or line in HEADING_STOPLIST or len(words) > 10:
        return False
    if sum(c.isalpha() for c in line) < 4 or (len(words) > 1 and len(set(words)) == 1):
        return False
    if any(rx.search(line) for _, rx in PART_MARKERS):
        return False
    return n_numbers(line) == 0


def page_is_tabular(lines: List[str]) -> bool:
    """Figure/table pages: dominated by numeric rows or short chart fragments."""
    if not lines:
        return True
    rows = sum(is_table_row(l) for l in lines)
    short = sum(len(l) < 45 for l in lines)
    has_title = any(TITLE_LINE.match(l) for l in lines)
    return rows / len(lines) > 0.3 or (has_title and short / len(lines) > 0.55)


def is_sentence_like(line: str) -> bool:
    return len(line.split()) >= 6 and bool(re.search(r"[a-z]", line)) and not is_table_row(line)


@dataclass
class Section:
    sid: str
    part: str
    title: str
    start_page: int
    end_page: int
    prose: List[Tuple[int, str]] = field(default_factory=list)      # (page, line)
    tables: List[Tuple[int, str]] = field(default_factory=list)     # (page, layout line)
    subsections: List[str] = field(default_factory=list)

    @property
    def prose_text(self) -> str:
        return "\n\n".join(p for _, p in reflow(self.prose))

    @property
    def n_words(self) -> int:
        return len(self.prose_text.split())


def reflow(lines: List[Tuple[int, str]]) -> List[Tuple[int, str]]:
    """Join hard-wrapped lines into paragraphs; IMF numbered paragraphs, bullets and headings start new ones."""
    paras, buf, buf_page = [], "", None
    new_block = re.compile(r"^(?:\d{1,2}\.\s+[A-Z]|[•▪]\s*|\(?[a-z]\)\s|[A-H]\.\s+[A-Z])")
    for page, line in lines:
        if new_block.match(line) or is_upper_heading(line) or TITLE_LINE.match(line):
            if buf:
                paras.append((buf_page, buf))
            buf, buf_page = line, page
            continue
        if not buf:
            buf, buf_page = line, page
        elif buf.endswith("-") and line[:1].islower():
            buf += line                                   # keep the hyphen: 'climate-' + 'resilient'
        else:
            buf += " " + line
    if buf:
        paras.append((buf_page, buf))
    return paras


def build_sections(pages: List[Dict[str, Any]]) -> List[Section]:
    sections: List[Section] = []
    part, cur = "Cover", None

    def open_section(title: str, page: int, part_name: str) -> Section:
        s = Section(sid=f"S{len(sections) + 1:02d}", part=part_name, title=title, start_page=page, end_page=page)
        sections.append(s)
        return s

    for pg in pages:
        lines, page_no = pg["lines"], pg["page"]
        new_part = detect_part("\n".join(lines)[:600], part)
        if new_part != part or cur is None:
            part = new_part
            cur = open_section(part, page_no, part)
        tabular = page_is_tabular(lines)
        if tabular:
            cur_tables = pg["layout_lines"]
            if part == "Staff Report" and not cur.title.startswith("Figures") and \
                    any(FIG_TAB_START.match(l) for l in lines):
                cur = open_section("Figures and Statistical Tables", page_no, part)
            cur.tables.extend((page_no, l) for l in cur_tables)
            cur.prose.extend((page_no, l) for l in lines if is_sentence_like(l))   # keep chart captions
            cur.end_page = page_no
            continue
        i = 0
        while i < len(lines):
            line = lines[i]
            if part not in WHOLE_PARTS:
                m = ANNEX_HEADING.match(line)
                if m:
                    if not cur.title.startswith(f"Annex {m.group(1)}."):
                        cur = open_section(f"Annex {m.group(1)}. {m.group(2).rstrip('0123456789')}", page_no, part)
                    i += 1
                    continue
                if is_upper_heading(line):
                    title = line
                    while i + 1 < len(lines) and is_upper_heading(lines[i + 1]):   # two-line headings
                        i += 1
                        title += " " + lines[i]
                    cur = open_section(title.title(), page_no, part)
                    i += 1
                    continue
                if part == "Staff Report" and FIG_TAB_START.match(line) and not cur.title.startswith("Figures"):
                    cur = open_section("Figures and Statistical Tables", page_no, part)
            if re.match(r"^[A-H]\.\s+[A-Z][a-z]", line) and len(line) < 110:
                cur.subsections.append(line)
            if is_table_row(line):
                cur.tables.append((page_no, line))
            else:
                cur.prose.append((page_no, line))
            cur.end_page = page_no
            i += 1
    # merge tiny fragments (e.g. a heading immediately followed by another) into the next section
    merged: List[Section] = []
    for s in sections:
        if merged and merged[-1].n_words < 40 and not merged[-1].tables and merged[-1].part == s.part \
                and merged[-1].part not in WHOLE_PARTS:
            prev = merged.pop()
            s.prose = prev.prose + s.prose
            s.start_page = prev.start_page
        merged.append(s)
    for k, s in enumerate(merged, start=1):
        s.sid = f"S{k:02d}"
    return merged


SECTIONS = build_sections(PAGES)
section_df = pd.DataFrame([{"sid": s.sid, "part": s.part, "title": s.title, "pages": f"{s.start_page}-{s.end_page}",
                            "prose_words": s.n_words, "table_lines": len(s.tables),
                            "subsections": "; ".join(s.subsections[:4])} for s in SECTIONS])
display(section_df)
save_json([{**asdict(s), "prose_text": s.prose_text} for s in SECTIONS], "sections.json")

### 1.4 Token-aware chunking + vector index
* **Token counting** uses the *embedding model's own tokenizer*. A chunk's token count is then exactly what the embedder sees, and every chunk fits its 512-token window. If that tokenizer is unavailable, `tiktoken` takes over, then a regex estimate.
* **Prose chunks** are built from whole sentences, target **400 tokens**, with a sentence-aligned **~50-token overlap**. A fact on a chunk boundary therefore appears complete in at least one chunk. Chunks never cross a section boundary, and each carries its section and page range for citation.
* **Table chunks** come from layout-mode text. When a table spans several chunks, its **title and year header are repeated** at the top of each continuation chunk, so a row like `Regulatory capital ratio 27.51` never loses its "Commercial Banks, 2013–23" context.

The FAISS index is built here, ahead of Task 3, because Task 2's indicator extraction already uses retrieval for grounding. Task 3 then evaluates retrieval on its own.

In [ ]:
class TokenCounter:
    """Embedding-model tokenizer → tiktoken → regex estimate (first that loads wins)."""
    def __init__(self, hf_name: str):
        try:
            from transformers import AutoTokenizer, logging as hf_logging
            hf_logging.set_verbosity_error()
            tok = AutoTokenizer.from_pretrained(hf_name)
            self._fn = lambda s: len(tok.encode(s, add_special_tokens=False))
            self.name = f"{hf_name} tokenizer"
        except Exception:                                     # noqa: BLE001
            try:
                import tiktoken
                enc = tiktoken.get_encoding("cl100k_base")
                self._fn = lambda s: len(enc.encode(s))
                self.name = "tiktoken cl100k_base"
            except Exception:                                 # noqa: BLE001
                self._fn = lambda s: int(len(re.findall(r"\w+|[^\w\s]", s)) * 1.15)
                self.name = "regex estimate"

    def __call__(self, text: str) -> int:
        return self._fn(text)


count_tokens = TokenCounter(CONFIG["embedding_model"])
print("Token counter:", count_tokens.name)

ABBREVIATIONS = ["e.g.", "i.e.", "U.S.", "U.K.", "No.", "Nos.", "Mr.", "Ms.", "Mrs.", "Dr.", "vs.", "etc.",
                 "Fig.", "Vol.", "St.", "Jan.", "Feb.", "Mar.", "Apr.", "Aug.", "Sept.", "Oct.", "Nov.", "Dec.",
                 "approx.", "incl.", "est."]


def split_sentences(text: str) -> List[str]:
    protected = text
    for a in ABBREVIATIONS:
        protected = protected.replace(a, a.replace(".", "<DOT>"))
    parts = [p.replace("<DOT>", ".").strip()
             for p in re.split(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-Z0-9•])", protected) if p.strip()]
    merged: List[str] = []
    for p in parts:                       # re-attach IMF paragraph numbers such as "12."
        if merged and len(merged[-1]) <= 4:
            merged[-1] = merged[-1] + " " + p
        else:
            merged.append(p)
    return merged


@dataclass
class Chunk:
    chunk_id: str
    sid: str
    section: str
    part: str
    kind: str                 # "prose" | "table"
    page_start: int
    page_end: int
    text: str
    n_tokens: int

    @property
    def embed_text(self) -> str:
        return f"{self.part} | {self.section}: {self.text}"

    @property
    def cite(self) -> str:
        pages = f"p.{self.page_start}" if self.page_start == self.page_end else f"pp.{self.page_start}-{self.page_end}"
        return f"[{self.chunk_id} | {self.section} | {pages}]"


def _split_long(page: int, sent: str, target: int) -> List[Tuple[int, str]]:
    words, out, buf = sent.split(), [], []
    for w in words:
        buf.append(w)
        if count_tokens(" ".join(buf)) >= target:
            out.append((page, " ".join(buf)))
            buf = []
    if buf:
        out.append((page, " ".join(buf)))
    return out


def chunk_units(units: List[Tuple[int, str]], target: int, overlap: int, hard_max: int) -> List[List[Tuple[int, str]]]:
    """Greedy sentence packing with a sentence-aligned (word-level fallback) token overlap."""
    expanded: List[Tuple[int, str]] = []
    for page, s in units:
        expanded.extend(_split_long(page, s, target // 2) if count_tokens(s) > target // 2 else [(page, s)])
    chunks, cur, cur_tok = [], [], 0
    for page, s in expanded:
        t = count_tokens(s)
        if cur and cur_tok + t > target:
            chunks.append(cur)
            tail, tail_tok = [], 0
            for u in reversed(cur):
                ut = count_tokens(u[1])
                if tail_tok + ut > overlap:
                    break
                tail.insert(0, u)
                tail_tok += ut
            if not tail:                                   # last sentence too long → word-level overlap
                last_page, last = cur[-1]
                n_words = max(5, int(overlap / 1.3))
                tail = [(last_page, " ".join(last.split()[-n_words:]))]
                tail_tok = count_tokens(tail[0][1])
            cur, cur_tok = tail, tail_tok
        cur.append((page, s))
        cur_tok += t
    if cur:
        chunks.append(cur)
    return chunks


def chunk_section_prose(sec: Section) -> List[Chunk]:
    units = [(page, sent) for page, para in reflow(sec.prose) for sent in split_sentences(para)]
    groups = chunk_units(units, CONFIG["chunk_target_tokens"], CONFIG["chunk_overlap_tokens"],
                         CONFIG["chunk_hard_max_tokens"])
    out = []
    for g in groups:
        text = " ".join(s for _, s in g)
        n = count_tokens(text)
        if n < CONFIG["chunk_min_tokens"] and out:          # fold a tiny tail into the previous chunk
            prev = out[-1]
            if prev.n_tokens + n <= CONFIG["chunk_hard_max_tokens"]:
                prev.text += " " + text
                prev.n_tokens = count_tokens(prev.text)
                prev.page_end = g[-1][0]
                continue
        out.append(Chunk("", sec.sid, sec.title, sec.part, "prose", g[0][0], g[-1][0], text, n))
    return out


def chunk_section_tables(sec: Section) -> List[Chunk]:
    out, buf, buf_pages, title, header = [], [], [], "", ""

    def flush(force_context: bool):
        nonlocal buf, buf_pages
        body = [l for l in buf if l.strip()]
        if body and any(n_numbers(l) >= 2 for l in body):
            text = "\n".join(body)
            out.append(Chunk("", sec.sid, sec.title, sec.part, "table", min(buf_pages), max(buf_pages),
                             text, count_tokens(text)))
        buf, buf_pages = [], []
        if force_context:                                     # continuation chunk keeps its context
            buf = [l for l in (f"{title} (continued)" if title else "", header) if l]

    for page, line in sec.tables:
        if TITLE_LINE.match(line):
            if buf:
                flush(force_context=False)
            title, header = line, ""
        elif len(years_in(line)) >= 4:
            header = line
        if count_tokens("\n".join(buf + [line])) > CONFIG["chunk_target_tokens"]:
            flush(force_context=True)
        buf.append(line)
        buf_pages.append(page)
    if buf:
        flush(force_context=False)
    return out


def years_in(line: str) -> List[int]:
    return [int(t) for t in re.findall(r"(?<![\d.])(19[89]\d|20[0-5]\d)(?![\d.])", line)]


CHUNKS: List[Chunk] = []
for sec in SECTIONS:
    if sec.part == "Cover":
        continue
    CHUNKS.extend(chunk_section_prose(sec) + chunk_section_tables(sec))
for k, c in enumerate(CHUNKS, start=1):
    c.chunk_id = f"C{k:03d}"
CHUNK_BY_ID = {c.chunk_id: c for c in CHUNKS}

chunk_df = pd.DataFrame([{**asdict(c)} for c in CHUNKS])
print(f"{len(CHUNKS)} chunks ({(chunk_df.kind == 'prose').sum()} prose, {(chunk_df.kind == 'table').sum()} table)")
display(chunk_df.groupby("kind")["n_tokens"].describe().round(1))
over = chunk_df[chunk_df.n_tokens > CONFIG["chunk_hard_max_tokens"]]
print(f"Chunks above the {CONFIG['chunk_hard_max_tokens']}-token hard limit: {len(over)}")

# Overlap check: share of consecutive prose chunks (same section) that share text at the boundary
pairs = [(a, b) for a, b in zip(CHUNKS, CHUNKS[1:]) if a.kind == b.kind == "prose" and a.sid == b.sid]
shared = sum(any(w in b.text[:400] for w in [a.text[-60:].strip()[:40]]) for a, b in pairs)
print(f"Consecutive prose-chunk pairs with boundary overlap: {shared}/{len(pairs)}")
with open(OUT_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
    for c in CHUNKS:
        f.write(json.dumps(asdict(c), ensure_ascii=False) + "\n")

In [ ]:
class VectorIndex:
    """Dense retrieval: sentence-transformers embeddings (unit-normalised) + FAISS IndexFlatL2.
    On unit vectors, squared L2 distance = 2 − 2·cosine, so L2 ranking equals cosine ranking."""

    def __init__(self, model_name: str, query_prefix: str = ""):
        self.model = SentenceTransformer(model_name)
        self.model_name, self.query_prefix = model_name, query_prefix
        self.index, self.chunks, self.emb = None, [], None

    def build(self, chunks: List[Chunk]) -> "VectorIndex":
        texts = [c.embed_text for c in chunks]
        key = hashlib.sha256((self.model_name + "\x00".join(texts)).encode("utf-8")).hexdigest()[:20]
        path = CACHE_DIR / "embeddings" / f"{key}.npy"
        if path.exists():
            emb = np.load(path)
        else:
            emb = self.model.encode(texts, batch_size=32, normalize_embeddings=True,
                                    show_progress_bar=False).astype("float32")
            np.save(path, emb)
        self.index = faiss.IndexFlatL2(emb.shape[1])
        self.index.add(emb)
        self.chunks, self.emb = chunks, emb
        return self

    def encode_query(self, query: str) -> np.ndarray:
        return self.model.encode([self.query_prefix + query], normalize_embeddings=True).astype("float32")

    def search(self, query: str, k: int = 5, kind: Optional[str] = None,
               parts: Optional[Tuple[str, ...]] = None) -> List[Tuple[Chunk, float]]:
        """Return (chunk, cosine similarity) pairs, optionally filtered by chunk kind / document part."""
        n_probe = len(self.chunks) if (kind or parts) else min(k, len(self.chunks))
        dist, idx = self.index.search(self.encode_query(query), n_probe)
        hits = []
        for d, i in zip(dist[0], idx[0]):
            if i < 0:
                continue
            c = self.chunks[i]
            if (kind and c.kind != kind) or (parts and c.part not in parts):
                continue
            hits.append((c, float(1 - d / 2)))
            if len(hits) == k:
                break
        return hits


def format_context(hits: List[Tuple[Chunk, float]], max_tokens: int = 14_000) -> str:
    """Numbered, citable context block for grounded prompts (deduplicated, page-ordered)."""
    seen, blocks, total = set(), [], 0
    for c, _ in sorted(hits, key=lambda h: (h[0].page_start, h[0].chunk_id)):
        if c.chunk_id in seen:
            continue
        seen.add(c.chunk_id)
        total += c.n_tokens
        if total > max_tokens:
            break
        blocks.append(f"{c.cite} ({c.kind})\n{c.text}")
    return "\n\n".join(blocks)


VINDEX = VectorIndex(CONFIG["embedding_model"], CONFIG["embedding_query_prefix"]).build(CHUNKS)
print(f"FAISS IndexFlatL2: {VINDEX.index.ntotal} vectors × {VINDEX.emb.shape[1]} dims")
for c, s in VINDEX.search("bank nonperforming loans and provisioning", k=3):
    print(f"  {s:.3f} {c.cite} {c.text[:110]}…")

### 1.5 Prompt library and output schemas
All prompts are kept in one place and saved to `outputs/prompts.json` so the report can quote them exactly. Design choices:
* **Role + grounding rule in the system instruction.** "Use only the provided text", with an explicit guard against confusing **Dominica** with **the Dominican Republic**. This report itself writes "the Dominican economy" when it means Dominica, which makes the mix-up easy.
* **Quantified constraints.** Word limits and bullet counts are stated as numbers, then **verified in code**. The LLM's claim to have complied is never trusted.
* **Evidence fields.** Structured outputs must carry a verbatim `evidence_quote`, which is fuzzy-matched against the source to catch fabricated quotes.
* **Judge rubric with anchors.** Each 1–5 score has a defined 1/3/5 meaning, which cuts down score inflation and makes judge runs comparable.

In [ ]:
C = CONFIG["country"]
SYSTEM_ANALYST = (
    f"You are a senior IMF macro-financial analyst summarising an IMF Article IV report on {C}. "
    "Work strictly from the source text supplied in the prompt: never use outside knowledge, never guess, "
    f"and never confuse {C} with {CONFIG['confusable_country']} (the report may say 'the Dominican economy' "
    f"when it means {C}). Reproduce numbers, units and years exactly as written."
)
SYSTEM_JUDGE = (
    "You are an independent, strict evaluator auditing AI-generated text against a source document. "
    "You did not write the candidate. Judge only what the source supports; do not reward fluency. "
    "Use the full 1-5 scale and the rubric anchors exactly."
)

SECTION_SUMMARY_PROMPT = """Summarise the following section of the IMF {doc} for {country}.
Constraints:
- At most {target} words (hard limit: fewer than {limit} words). One paragraph, no bullets, no headings.
- Lead with the section's main conclusion, then the 2-3 most decision-relevant facts (keep figures with their years and units) and any policy recommendation or risk.
- Use only information in the section text. If the text is mostly chart captions or table rows, summarise what they show.
- Do not begin with "This section" or repeat the title.

Section title: {title}
Section text:
<<<
{text}
>>>"""

REFINE_SUFFIX = """

An independent reviewer audited your previous draft.
Previous draft: {draft}
Reviewer feedback: {feedback}
Unsupported claims to remove: {unsupported}
Missing points to include if space allows: {missing}
Write an improved summary that fixes these issues and still obeys every constraint above."""

COMPRESS_PROMPT = """Rewrite the summary below in at most {target} words. Keep the key figures; add nothing new.
Summary: {text}"""

SUMMARY_JUDGE_PROMPT = """Evaluate the CANDIDATE summary against the SOURCE text.
Score each criterion with an integer from 1 to 5:
- consistency: 5 = no contradiction with the source and internally coherent; 3 = minor imprecision (e.g. rounded or mis-dated figure); 1 = contradicts the source.
- completeness: 5 = captures the main conclusion and the most decision-relevant facts; 3 = misses one key point; 1 = misses most key points.
- factual_alignment: 5 = every number, date and claim is traceable to the source; 3 = one unsupported or mis-attributed claim; 1 = several hallucinated claims.
List every unsupported claim (quote it) and up to 3 important missing points.
verdict = "pass" only if every score >= {threshold} and there are no unsupported claims, otherwise "revise".
Constraint context: {constraint_note}

SOURCE:
<<<
{source}
>>>

CANDIDATE:
<<<
{candidate}
>>>"""

EXEC_SUMMARY_PROMPT = """Write an executive summary of the COMPLETE IMF Article IV report on {country} for a financial-stability policymaker.
Return 6-10 bullets. Each bullet is ONE sentence (max 35 words) stating a primary financial-stability finding or risk assessment, with its key figure and year where the source gives one.
Cover, where the report addresses them: growth outlook; inflation; fiscal position and public debt; banking-sector soundness (capital, NPLs, provisioning); non-bank / credit-union risks; external position; main downside risks and the IMF's headline recommendation.
For each bullet also return: theme (one of: {themes}; or "Growth & Outlook"; or "Fiscal & External") and evidence_quote copied VERBATIM from the source (max 30 words).

SOURCE:
<<<
{text}
>>>"""



class SummaryJudgement(BaseModel):
    consistency: int = Field(description="1-5")
    completeness: int = Field(description="1-5")
    factual_alignment: int = Field(description="1-5")
    unsupported_claims: List[str] = Field(default_factory=list)
    missing_points: List[str] = Field(default_factory=list)
    verdict: str = Field(description="pass or revise")
    feedback: str = ""


class ExecBullet(BaseModel):
    point: str
    theme: str
    evidence_quote: str


class ExecutiveSummary(BaseModel):
    bullets: List[ExecBullet]


PROMPTS: Dict[str, str] = {"SYSTEM_ANALYST": SYSTEM_ANALYST, "SYSTEM_JUDGE": SYSTEM_JUDGE,
                           "SECTION_SUMMARY_PROMPT": SECTION_SUMMARY_PROMPT, "REFINE_SUFFIX": REFINE_SUFFIX,
                           "COMPRESS_PROMPT": COMPRESS_PROMPT, "SUMMARY_JUDGE_PROMPT": SUMMARY_JUDGE_PROMPT,
                           "EXEC_SUMMARY_PROMPT": EXEC_SUMMARY_PROMPT}

### 1.6 Quote verification (deterministic hallucination check)
A quote counts as **verified** when ≥ 80% of its word 4-grams occur in the cleaned source. This tolerates whitespace and punctuation differences from PDF extraction but rejects paraphrased or invented "quotes".

In [ ]:
def _norm_words(text: str) -> List[str]:
    return re.findall(r"[a-z0-9.%]+", normalise_text(text).lower())


FULL_TEXT = "\n\n".join(s.prose_text + "\n" + "\n".join(l for _, l in s.tables) for s in SECTIONS)
_SOURCE_WORDS = _norm_words(FULL_TEXT)
SOURCE_4GRAMS = {tuple(_SOURCE_WORDS[i:i + 4]) for i in range(len(_SOURCE_WORDS) - 3)}


def quote_support(quote: str) -> float:
    w = _norm_words(quote or "")
    if len(w) < 4:
        return 1.0 if w and " ".join(w) in " ".join(_SOURCE_WORDS) else 0.0
    grams = [tuple(w[i:i + 4]) for i in range(len(w) - 3)]
    return sum(g in SOURCE_4GRAMS for g in grams) / len(grams)


def quote_verified(quote: str, threshold: float = 0.8) -> bool:
    return quote_support(quote) >= threshold


print("Sanity check – real sentence:", round(quote_support(" ".join(_SOURCE_WORDS[500:530])), 2),
      "| invented sentence:", round(quote_support("Dominica's central bank raised its policy rate to 7 percent in 2023"), 2))

### 1.7 Section summaries with the generate → judge → refine loop
For every major section:
1. The **extractor** writes a summary under 100 words. Sections above `section_max_tokens` are summarised map-reduce style.
2. A **code check** counts the words. A summary over the limit is compressed once and the violation is logged.
3. The **judge** (a different model) scores consistency, completeness and factual alignment against the section text.
4. If the verdict is `revise`, the judge's feedback is sent back to the extractor for one refinement round, and the new draft is **re-judged**. Both rounds are kept, so the report can show whether feedback actually helps.

In [ ]:
def word_count(text: str) -> int:
    return len(re.findall(r"\b[\w'.%-]+\b", text))


def section_source(sec: Section, table_token_budget: int = 1500) -> str:
    """Prose plus a token-capped slice of table rows (for chart/table-heavy sections)."""
    src = sec.prose_text
    if sec.tables and sec.n_words < 400:
        rows, used = [], 0
        for _, line in sec.tables:
            used += count_tokens(line)
            if used > table_token_budget:
                break
            rows.append(line)
        src += "\n\n[Table extracts]\n" + "\n".join(rows)
    return src


def judge_summary(source: str, candidate: str, tag: str, constraint_note: Optional[str] = None) -> Optional[SummaryJudgement]:
    note = constraint_note or (f"the word limit is fewer than {CONFIG['summary_word_limit']} words and the "
                               f"candidate has {word_count(candidate)} words.")
    prompt = SUMMARY_JUDGE_PROMPT.format(threshold=CONFIG["judge_pass_threshold"], constraint_note=note,
                                         source=source, candidate=candidate)
    res = llm.generate(prompt, JUDGE_MODEL, system=SYSTEM_JUDGE, schema=SummaryJudgement, tag=tag)
    if not res.ok:
        print(f"  judge failed for {tag}: {res.error}")
        return None
    j = res.parsed
    for f_ in ("consistency", "completeness", "factual_alignment"):        # clamp to the rubric range
        setattr(j, f_, int(min(5, max(1, getattr(j, f_)))))
    return j


def judge_passes(j: Optional[SummaryJudgement]) -> bool:
    if j is None:
        return True                                        # nothing to act on – logged as missing score
    t = CONFIG["judge_pass_threshold"]
    return min(j.consistency, j.completeness, j.factual_alignment) >= t and not j.unsupported_claims


def generate_summary(title: str, text: str, doc: str, tag: str) -> Tuple[str, Dict[str, Any]]:
    """Single summarisation call (with map-reduce for very long inputs) + word-limit repair."""
    meta = {"map_reduce": False, "compressed": False}
    if count_tokens(text) > CONFIG["section_max_tokens"]:
        meta["map_reduce"] = True
        paras, parts, buf = text.split("\n\n"), [], ""
        for p in paras:
            if buf and count_tokens(buf + p) > CONFIG["section_max_tokens"] // 2:
                parts.append(buf)
                buf = ""
            buf += p + "\n\n"
        parts.append(buf)
        partials = []
        for k, part in enumerate(parts):
            r = llm.generate(SECTION_SUMMARY_PROMPT.format(doc=doc, country=C, target=120, limit=150,
                                                           title=f"{title} (part {k + 1})", text=part),
                             EXTRACTOR_MODEL, system=SYSTEM_ANALYST, tag=f"{tag}-map{k}")
            partials.append(r.text.strip())
        text = "\n\n".join(partials)
    res = llm.generate(SECTION_SUMMARY_PROMPT.format(doc=doc, country=C, target=CONFIG["summary_target_words"],
                                                     limit=CONFIG["summary_word_limit"], title=title, text=text),
                       EXTRACTOR_MODEL, system=SYSTEM_ANALYST, tag=tag)
    summary = res.text.strip()
    meta.update({"latency_s": res.latency_s, "error": res.error, "words_initial": word_count(summary)})
    if word_count(summary) >= CONFIG["summary_word_limit"]:
        meta["compressed"] = True
        summary = llm.generate(COMPRESS_PROMPT.format(target=CONFIG["summary_target_words"] - 10, text=summary),
                               EXTRACTOR_MODEL, system=SYSTEM_ANALYST, tag=f"{tag}-compress").text.strip()
    return summary, meta


def summarise_with_judge(title: str, source: str, doc: str, tag: str) -> Dict[str, Any]:
    summary, meta = generate_summary(title, source, doc, tag)
    j = judge_summary(source, summary, f"{tag}-judge0")
    rounds = [{"round": 0, "summary": summary, "words": word_count(summary),
               "judge": j.model_dump() if j else None}]
    for r in range(1, CONFIG["max_refinement_rounds"] + 1):
        if judge_passes(j):
            break
        prompt = SECTION_SUMMARY_PROMPT.format(doc=doc, country=C, target=CONFIG["summary_target_words"],
                                               limit=CONFIG["summary_word_limit"], title=title, text=source)
        prompt += REFINE_SUFFIX.format(draft=summary, feedback=j.feedback, unsupported=j.unsupported_claims or "none",
                                       missing=j.missing_points or "none")
        summary = llm.generate(prompt, EXTRACTOR_MODEL, system=SYSTEM_ANALYST, tag=f"{tag}-refine{r}").text.strip()
        if word_count(summary) >= CONFIG["summary_word_limit"]:
            summary = llm.generate(COMPRESS_PROMPT.format(target=CONFIG["summary_target_words"] - 10, text=summary),
                                   EXTRACTOR_MODEL, system=SYSTEM_ANALYST, tag=f"{tag}-refine{r}-compress").text.strip()
        j = judge_summary(source, summary, f"{tag}-judge{r}")
        rounds.append({"round": r, "summary": summary, "words": word_count(summary),
                       "judge": j.model_dump() if j else None})
    return {"final_summary": summary, "final_words": word_count(summary),
            "within_limit": word_count(summary) < CONFIG["summary_word_limit"], "rounds": rounds, **meta}


SUMMARISABLE = [s for s in SECTIONS if s.part != "Cover" and (s.n_words >= 80 or s.tables)]
SECTION_SUMMARIES: List[Dict[str, Any]] = []
for sec in SUMMARISABLE:
    doc = sec.part if sec.part in WHOLE_PARTS else "Staff Report"
    out = summarise_with_judge(sec.title, section_source(sec), doc, tag=f"summary-{sec.sid}")
    SECTION_SUMMARIES.append({"sid": sec.sid, "part": sec.part, "title": sec.title,
                              "pages": f"{sec.start_page}-{sec.end_page}", "source_words": sec.n_words, **out})
    print(f"{sec.sid} {sec.title[:45]:<45} words={out['final_words']:>3} rounds={len(out['rounds'])}")
save_json(SECTION_SUMMARIES, "summaries.json")


def judge_rows(items: List[Dict[str, Any]], kind: str) -> pd.DataFrame:
    rows = []
    for it in items:
        for rd in it["rounds"]:
            if rd["judge"]:
                rows.append({"kind": kind, "id": it["sid"], "title": it["title"], "round": rd["round"],
                             "words": rd["words"], **{k: rd["judge"][k] for k in
                                                     ("consistency", "completeness", "factual_alignment")},
                             "n_unsupported": len(rd["judge"]["unsupported_claims"]),
                             "verdict": rd["judge"]["verdict"]})
    return pd.DataFrame(rows)


SUMMARY_JUDGE_DF = judge_rows(SECTION_SUMMARIES, "section_summary")
display(pd.DataFrame([{"Section": s["title"], "Pages": s["pages"], "Words": s["final_words"],
                       "Summary": s["final_summary"]} for s in SECTION_SUMMARIES]))
if not SUMMARY_JUDGE_DF.empty:
    display(SUMMARY_JUDGE_DF.groupby("round")[["consistency", "completeness", "factual_alignment",
                                                "n_unsupported"]].mean().round(2))
print("Word-limit compliance:", sum(s["within_limit"] for s in SECTION_SUMMARIES), "/", len(SECTION_SUMMARIES),
      "| needed compression:", sum(s["compressed"] for s in SECTION_SUMMARIES))

### 1.8 Executive summary of the full report
The narrative of the whole package (press release, staff report and annexes, ED statement, ≈ 40–60k tokens) fits comfortably in Gemini's context window. The executive summary is therefore generated from the **full source text**, not by summarising summaries. That avoids the information loss of map-reduce. If the text ever exceeded `exec_summary_max_input_tokens`, the code would fall back to the section summaries. Each bullet's evidence quote is checked deterministically, and the judge audits the whole summary.

In [ ]:
EXEC_SOURCE = "\n\n".join(f"## {s.title} ({s.part}, pp.{s.start_page}-{s.end_page})\n{s.prose_text}"
                          for s in SECTIONS if s.part != "Cover" and s.n_words > 0)
exec_input_tokens = count_tokens(EXEC_SOURCE)
if exec_input_tokens > CONFIG["exec_summary_max_input_tokens"]:
    EXEC_SOURCE = "\n\n".join(f"## {s['title']}\n{s['final_summary']}" for s in SECTION_SUMMARIES)
    exec_mode = "section-summaries (map-reduce fallback)"
else:
    exec_mode = "full narrative text"
print(f"Executive-summary input: {exec_mode}, ~{count_tokens(EXEC_SOURCE):,} tokens")

exec_res = llm.generate(EXEC_SUMMARY_PROMPT.format(country=C, themes="; ".join(THEMES), text=EXEC_SOURCE),
                        EXTRACTOR_MODEL, system=SYSTEM_ANALYST, schema=ExecutiveSummary, tag="exec-summary")
EXEC_BULLETS = exec_res.parsed.bullets if exec_res.ok else []
if not exec_res.ok:
    print("Executive summary failed:", exec_res.error)
exec_text = "\n".join(f"- {b.point}" for b in EXEC_BULLETS)
exec_judge = judge_summary(EXEC_SOURCE, exec_text, "exec-judge",
                           constraint_note=f"6-10 one-sentence bullets of <= 35 words; the candidate has {len(EXEC_BULLETS)} bullets."
                           ) if EXEC_BULLETS else None
EXEC_SUMMARY = {
    "mode": exec_mode, "model": EXTRACTOR_MODEL, "latency_s": exec_res.latency_s,
    "n_bullets": len(EXEC_BULLETS), "bullet_count_ok": 6 <= len(EXEC_BULLETS) <= 10,
    "bullets": [{**b.model_dump(), "words": word_count(b.point),
                 "quote_support": round(quote_support(b.evidence_quote), 3),
                 "quote_verified": quote_verified(b.evidence_quote)} for b in EXEC_BULLETS],
    "judge": exec_judge.model_dump() if exec_judge else None,
}
save_json(EXEC_SUMMARY, "executive_summary.json")
display(Markdown("**Executive summary**\n\n" + exec_text))
print("Quotes verified:", sum(b["quote_verified"] for b in EXEC_SUMMARY["bullets"]), "/", len(EXEC_BULLETS))
if exec_judge:
    print("Judge:", {k: getattr(exec_judge, k) for k in ("consistency", "completeness", "factual_alignment", "verdict")})

---
## Task 2 — Thematic extraction, indicators and structured output
### 2.1 Thematic distribution across the 7 financial-stability themes
Two complementary measures are computed:
1. **Lexicon mention counts.** Each theme has a curated set of regex patterns (multi-word terms included). Counts are reported raw and **per 1,000 words** so the result doesn't depend on document length. The lexicon is transparent and reproducible, but it can over-count generic words like *bank* or *credit*.
2. **Embedding-based chunk classification.** Each prose chunk goes to the theme whose description is nearest in embedding space. The bottom decile of best-match similarity is labelled "Other". This captures what a chunk is *about*, not which words it repeats.

Where the two measures disagree is itself worth discussing in the report: it exposes lexicon bias.

In [ ]:
THEME_LEXICON: Dict[str, List[str]] = {
    "Inflation & Prices": [r"inflation\w*", r"disinflation", r"consumer prices?", r"\bCPI\b", r"cost[- ]of[- ]living",
                           r"price (?:level|pressures?|increases?|shocks?|controls?|stability)",
                           r"(?:food|fuel|energy|commodity|oil|import) prices?", r"deflat\w*"],
    "Interest Rates & Credit": [r"interest rates?", r"policy rates?", r"discount rate", r"lending rates?",
                                r"deposit rates?", r"\bcredit\b(?!\s+unions?)", r"lending", r"\bloans?\b",
                                r"borrowing costs?", r"spreads?", r"financial intermediation", r"access to finance"],
    "Housing & Real Estate": [r"housing", r"real estate", r"mortgages?", r"propert(?:y|ies)", r"residential",
                              r"construction", r"house prices?", r"dwellings?", r"foreclosures?", r"\bland\b"],
    "Banking Capital & Liquidity": [r"capital adequacy", r"regulatory capital", r"capital (?:ratios?|buffers?|requirements?)",
                                    r"\btier[- ]?1\b", r"liquidity", r"liquid assets", r"non-?performing loans?",
                                    r"\bNPLs?\b", r"provision\w*", r"credit unions?", r"\bbanks?\b", r"banking",
                                    r"deposits?", r"profitability", r"return on (?:assets|equity)"],
    "Sovereign Debt": [r"public debt", r"government debt", r"sovereign", r"debt[- ]to[- ]GDP", r"debt sustainability",
                       r"\bDSA\b", r"debt distress", r"debt service", r"fiscal (?:rule|anchor|space|consolidation|buffers?)",
                       r"external debt", r"guaranteed debt", r"arrears", r"gross financing needs?"],
    "Labor Market Dynamics": [r"labou?r (?:market|force|mobility|productivity|participation)", r"unemploy\w*",
                              r"\bemployment\b", r"\bwages?\b", r"\bworkers?\b", r"\bjobs?\b", r"\bskills?\b",
                              r"\byouth\b", r"emigration", r"pensions?", r"social security"],
    "Systemic Financial Risk": [r"systemic", r"financial stability", r"financial (?:sector )?risks?", r"contagion",
                                r"spillovers?", r"stress tests?", r"macroprudential", r"interconnect\w*",
                                r"de-?risking", r"correspondent bank\w*", r"AML/CFT", r"money laundering",
                                r"\bcrisis\b", r"supervis\w*", r"resolution framework"],
}
THEME_DESCRIPTIONS = {
    "Inflation & Prices": "inflation, consumer prices, cost of living, food and fuel price pressures",
    "Interest Rates & Credit": "interest rates, bank lending and private sector credit growth, borrowing costs, access to finance",
    "Housing & Real Estate": "housing, real estate, mortgages, property markets and construction",
    "Banking Capital & Liquidity": "bank and credit union capital adequacy, liquidity, nonperforming loans and provisioning",
    "Sovereign Debt": "public debt, fiscal deficits, debt sustainability analysis, government financing and the fiscal rule",
    "Labor Market Dynamics": "employment, unemployment, wages, labor force participation, skills, pensions and social security",
    "Systemic Financial Risk": "systemic financial stability risks, contagion, supervision, AML/CFT, correspondent banking, stress tests",
}


def lexicon_counts(text: str) -> Dict[str, int]:
    out = {}
    for theme, pats in THEME_LEXICON.items():
        flags = 0
        out[theme] = sum(len(re.findall(p, text, flags=flags if re.search(r"[A-Z]{2,}", p) else re.IGNORECASE))
                         for p in pats)
    return out


PROSE_TEXT = "\n".join(s.prose_text for s in SECTIONS if s.part != "Cover")
n_words_total = len(PROSE_TEXT.split())
lex = lexicon_counts(PROSE_TEXT)

theme_emb = VINDEX.model.encode([THEME_DESCRIPTIONS[t] for t in THEMES], normalize_embeddings=True)
prose_idx = [i for i, c in enumerate(CHUNKS) if c.kind == "prose"]
sims = VINDEX.emb[prose_idx] @ theme_emb.T
# Forced-choice assignment; chunks in the bottom decile of best-match similarity are labelled "Other"
# (weakly thematic text such as mission logistics or energy-sector detail). A relative floor is used because
# absolute cosine levels differ across embedding models.
SIM_FLOOR = float(np.quantile(sims.max(axis=1), 0.10)) if len(prose_idx) else 0.0
chunk_theme = [THEMES[j] if sims[r, j] >= SIM_FLOOR else "Other" for r, j in enumerate(sims.argmax(axis=1))]
emb_counts = Counter(chunk_theme)

THEME_DF = pd.DataFrame({
    "theme": THEMES,
    "lexicon_mentions": [lex[t] for t in THEMES],
    "mentions_per_1k_words": [round(1000 * lex[t] / n_words_total, 2) for t in THEMES],
    "lexicon_share_pct": [round(100 * lex[t] / max(1, sum(lex.values())), 1) for t in THEMES],
    "embedding_chunks": [emb_counts.get(t, 0) for t in THEMES],
    "embedding_share_pct": [round(100 * emb_counts.get(t, 0) / max(1, len(prose_idx)), 1) for t in THEMES],
}).sort_values("lexicon_mentions", ascending=False)

per_part = pd.DataFrame({p: lexicon_counts("\n".join(s.prose_text for s in SECTIONS if s.part == p))
                         for p in PART_ORDER[1:] if any(s.part == p for s in SECTIONS)}).T
display(THEME_DF)
display(per_part)
print(f"Prose chunks labelled Other (best similarity < {SIM_FLOOR:.3f}, bottom decile): {emb_counts.get('Other', 0)}")
save_json({"method": {"lexicon": THEME_LEXICON, "embedding_descriptions": THEME_DESCRIPTIONS,
                      "similarity_floor": SIM_FLOOR, "prose_words": n_words_total},
           "distribution": THEME_DF.to_dict(orient="records"),
           "by_document_part": per_part.reset_index(names="part").to_dict(orient="records"),
           "chunk_assignments": [{"chunk_id": CHUNKS[i].chunk_id, "theme": t, "similarity": round(float(sims[r].max()), 3)}
                                 for r, (i, t) in enumerate(zip(prose_idx, chunk_theme))]}, "themes.json")

### 2.2 Qualitative risk insights: growth drivers and vulnerabilities
The extractor reads the full narrative and returns 5–7 **growth drivers / strengths** and 5–7 **systemic vulnerabilities / challenges**. Each one comes with a verbatim evidence quote. The pipeline then checks the item counts, verifies every quote against the source, and has the judge audit the list.

In [ ]:
INSIGHTS_PROMPT = """From the IMF Article IV report on {country} below, extract:
1. growth_drivers: 5-7 national macroeconomic growth drivers or strengths (e.g. resilient sectors, policy buffers, sound institutions).
2. vulnerabilities: 5-7 systemic vulnerabilities or challenges (e.g. fiscal, financial-sector, external, climate risks).
For each item: title (max 8 words), explanation (max 35 words, include a figure if the report gives one),
theme (one of: {themes}; or "Growth & Outlook"; or "Fiscal & External"; or "Climate & Structural"),
and evidence_quote copied VERBATIM from the source (max 30 words).
Items must be distinct and specific to {country}; do not list generic risks the report does not discuss.

SOURCE:
<<<
{text}
>>>"""
PROMPTS["INSIGHTS_PROMPT"] = INSIGHTS_PROMPT


class Insight(BaseModel):
    title: str
    explanation: str
    theme: str
    evidence_quote: str


class RiskInsights(BaseModel):
    growth_drivers: List[Insight]
    vulnerabilities: List[Insight]


ins_res = llm.generate(INSIGHTS_PROMPT.format(country=C, themes="; ".join(THEMES), text=EXEC_SOURCE),
                       EXTRACTOR_MODEL, system=SYSTEM_ANALYST, schema=RiskInsights, tag="risk-insights")
if ins_res.ok:
    INSIGHTS = ins_res.parsed
else:
    print("Insight extraction failed:", ins_res.error)
    INSIGHTS = RiskInsights(growth_drivers=[], vulnerabilities=[])


def insights_as_text(ins: RiskInsights) -> str:
    return "\n".join([f"STRENGTH: {i.title} - {i.explanation}" for i in ins.growth_drivers] +
                     [f"VULNERABILITY: {i.title} - {i.explanation}" for i in ins.vulnerabilities])


ins_judge = judge_summary(EXEC_SOURCE, insights_as_text(INSIGHTS), "insights-judge",
                          constraint_note="5-7 strengths and 5-7 vulnerabilities, each specific and supported by the source.")
RISK_INSIGHTS = {
    "model": EXTRACTOR_MODEL,
    "count_ok": {"growth_drivers": 5 <= len(INSIGHTS.growth_drivers) <= 7,
                 "vulnerabilities": 5 <= len(INSIGHTS.vulnerabilities) <= 7},
    **{k: [{**i.model_dump(), "quote_support": round(quote_support(i.evidence_quote), 3),
            "quote_verified": quote_verified(i.evidence_quote)} for i in getattr(INSIGHTS, k)]
       for k in ("growth_drivers", "vulnerabilities")},
    "judge": ins_judge.model_dump() if ins_judge else None,
}
save_json(RISK_INSIGHTS, "risk_insights.json")
for k in ("growth_drivers", "vulnerabilities"):
    display(pd.DataFrame(RISK_INSIGHTS[k])[["title", "theme", "explanation", "quote_verified"]]
            .rename(columns={"title": k.replace("_", " ").title()}))
if ins_judge:
    print("Judge:", {k: getattr(ins_judge, k) for k in ("consistency", "completeness", "factual_alignment", "verdict")})

### 2.3 Core indicator JSON: RAG-grounded, schema-constrained extraction
**Approach.** Each indicator has a precise **definition**, plus two retrieval queries (one targeting prose, one targeting tables). Their top-ranked chunks are merged into a single numbered context. The extractor then fills a Pydantic schema enforced through Gemini's `response_schema`. For every field it returns the value, reference year, status, a verbatim quote and the source chunk id. A separate **proxy** slot keeps a "close but not equal" measure (total regulatory capital vs Tier 1, say) out of the target field.

**Fallbacks.** Missing values become JSON `null`, not `-1.0`. A `-1.0` sentinel is a valid-looking number: it would pass type checks and then plot as a real negative rate. Each `null` carries a machine-readable reason (`not_reported`, `ambiguous`, `failed_validation`), which the report explains in domain terms.

**Sanity rules.** Plausible-range checks catch scale errors, such as 0.046 written instead of 4.6%.

In [ ]:
INDICATOR_SPECS: Dict[str, Dict[str, Any]] = {
    "gdp_growth_forecast_pct": {
        "definition": "Real GDP growth (annual % change) PROJECTED by IMF staff for the first forecast year (the year the report is published). Not a historical outturn.",
        "queries": ["real GDP growth projected outlook for this year", "Selected Economic Indicators real GDP annual percent change projected"],
        "range": (-30, 30)},
    "headline_inflation_rate_pct": {
        "definition": "Headline consumer price inflation, PERIOD-AVERAGE annual % change, for the latest actual or estimated year (not a projection). If only end-of-period is stated, use it and say so in note.",
        "queries": ["consumer prices inflation period average annual percent change", "inflation declined from its peak to percent"],
        "range": (-10, 60)},
    "central_bank_policy_rate_pct": {
        "definition": "The monetary-policy (policy/discount) rate set by the central bank responsible for the country; for a currency-union member, the common central bank's rate. Only if a numeric rate is explicitly stated.",
        "queries": ["central bank monetary policy rate interest rate", "currency union common central bank monetary policy and exchange rate arrangement"],
        "range": (-1, 40)},
    "non_performing_loan_ratio": {
        "definition": "Nonperforming loans to total gross loans (%) for COMMERCIAL BANKS, latest year. Do NOT use credit-union or system-wide figures for this field.",
        "queries": ["commercial banks nonperforming loans to total gross loans", "bank NPL ratio provisioning asset quality"],
        "range": (0, 100)},
    "unemployment_rate_pct": {
        "definition": "Unemployment rate (% of labour force), the latest value stated anywhere in the report; give its reference year.",
        "queries": ["unemployment rate labor market", "social indicators population unemployment rate literacy"],
        "range": (0, 60)},
    "tier_1_capital_ratio": {
        "definition": "Tier 1 capital to risk-weighted assets (%) for banks. If only a different capital measure is reported (e.g. total regulatory capital to risk-weighted assets), value must be null and that figure goes in proxy_value with proxy_definition.",
        "queries": ["tier 1 capital to risk-weighted assets banks", "regulatory capital ratio capital adequacy commercial banks"],
        "range": (0, 100)},
}
INDICATORS = list(INDICATOR_SPECS)


class IndicatorValue(BaseModel):
    value: Optional[float] = Field(None, description="number in percent units, or null if not reported")
    reference_year: Optional[str] = None
    status: str = Field(description="extracted | not_reported | ambiguous")
    source_quote: Optional[str] = Field(None, description="verbatim text (for tables: title + row label + value)")
    source_chunk: Optional[str] = Field(None, description="chunk id such as C012")
    proxy_value: Optional[float] = None
    proxy_definition: Optional[str] = None
    note: str = ""


class IndicatorExtraction(BaseModel):
    country_name: str
    gdp_growth_forecast_pct: IndicatorValue
    headline_inflation_rate_pct: IndicatorValue
    central_bank_policy_rate_pct: IndicatorValue
    non_performing_loan_ratio: IndicatorValue
    unemployment_rate_pct: IndicatorValue
    tier_1_capital_ratio: IndicatorValue


class CoreIndicators(BaseModel):
    """The flat target schema required by the brief."""
    country_name: str
    gdp_growth_forecast_pct: Optional[float]
    headline_inflation_rate_pct: Optional[float]
    central_bank_policy_rate_pct: Optional[float]
    non_performing_loan_ratio: Optional[float]
    unemployment_rate_pct: Optional[float]
    tier_1_capital_ratio: Optional[float]


INDICATOR_SYSTEM = (
    f"You are a precise data-extraction engine for IMF country reports on {C}. Rules: "
    "(1) Extract a value ONLY if it is explicitly stated in the CONTEXT for this country. "
    "(2) Never compute, convert, infer or use outside knowledge (even if you know the value). "
    "(3) If an indicator is not reported, set value=null, status='not_reported' and explain why in note. "
    "(4) Percentages are written in percent units (4.6 means 4.6%), never as fractions. "
    "(5) source_quote must be copied verbatim; for tables give the table title, the row label and the value. "
    "(6) Tables may list several institution types (e.g. commercial banks vs credit unions): use the row under the "
    "correct table title. "
    f"(7) Do not confuse {C} with {CONFIG['confusable_country']}."
)
INDICATOR_PROMPT = """Extract the core indicators for {country} from the CONTEXT passages of {report}.

Indicator definitions:
{definitions}

Return every field. Cite the chunk id (e.g. C012) that contains each value.

CONTEXT:
{context}"""
GENERIC_INDICATOR_PROMPT = """Here is text from an IMF report about {country}. Extract these indicators and return them as JSON with the keys country_name, gdp_growth_forecast_pct, headline_inflation_rate_pct, central_bank_policy_rate_pct, non_performing_loan_ratio, unemployment_rate_pct, tier_1_capital_ratio.

{context}"""
PROMPTS.update({"INDICATOR_SYSTEM": INDICATOR_SYSTEM, "INDICATOR_PROMPT": INDICATOR_PROMPT,
                "GENERIC_INDICATOR_PROMPT": GENERIC_INDICATOR_PROMPT})


def build_indicator_context() -> Tuple[str, List[str]]:
    hits = []
    for spec in INDICATOR_SPECS.values():
        for q in spec["queries"]:
            hits += VINDEX.search(q, k=3, kind="prose") + VINDEX.search(q, k=3, kind="table")
    ctx = format_context(hits, max_tokens=16_000)
    return ctx, sorted({c.chunk_id for c, _ in hits})


INDICATOR_CONTEXT, INDICATOR_CHUNKS = build_indicator_context()
DEFINITIONS = "\n".join(f"- {k}: {v['definition']}" for k, v in INDICATOR_SPECS.items())
print(f"Indicator context: {len(INDICATOR_CHUNKS)} chunks, ~{count_tokens(INDICATOR_CONTEXT):,} tokens")


def flatten_generic(raw: Any) -> Dict[str, Any]:
    """Map a free-form JSON reply onto the flat schema (accepts {'value': x} objects too)."""
    if not isinstance(raw, dict):
        return {}
    out = {}
    for k in ["country_name"] + INDICATORS:
        v = raw.get(k)
        if isinstance(v, dict):
            v = v.get("value")
        out[k] = v
    return out


def schema_compliance(flat: Dict[str, Any]) -> Dict[str, Any]:
    """Strict check: all keys present, numeric fields are float/int/null (no '4.6%' strings)."""
    missing = [k for k in ["country_name"] + INDICATORS if k not in flat]
    bad_types = [k for k in INDICATORS if k in flat and flat[k] is not None and not isinstance(flat[k], (int, float))]
    return {"valid_json": bool(flat), "missing_keys": missing, "bad_types": bad_types,
            "compliant": bool(flat) and not missing and not bad_types}


def run_indicator_extraction(model: str, style: str) -> Dict[str, Any]:
    """style = 'strict' (system instruction + response_schema) or 'generic' (plain user prompt, no schema)."""
    if style == "strict":
        res = llm.generate(INDICATOR_PROMPT.format(country=C, report=CONFIG["imf_country_report_no"],
                                                   definitions=DEFINITIONS, context=INDICATOR_CONTEXT),
                           model, system=INDICATOR_SYSTEM, schema=IndicatorExtraction, tag=f"indicators-{style}-{model}")
        detail = res.parsed.model_dump() if res.ok else {}
        flat = {"country_name": detail.get("country_name")} if detail else {}
        flat.update({k: detail[k]["value"] for k in INDICATORS if detail})
    else:
        res = llm.generate(GENERIC_INDICATOR_PROMPT.format(country=C, context=INDICATOR_CONTEXT),
                           model, tag=f"indicators-{style}-{model}")
        detail, flat = {}, flatten_generic(extract_json_block(res.text))
    return {"model": model, "style": style, "latency_s": round(res.latency_s, 2), "cached": res.cached,
            "error": res.error, "flat": flat, "detail": detail, "compliance": schema_compliance(flat)}


def apply_fallbacks(run: Dict[str, Any]) -> Tuple[CoreIndicators, Dict[str, Dict[str, Any]]]:
    """Coerce to the target schema; every null gets an explicit, machine-readable fallback reason."""
    flat, detail, reasons = dict(run["flat"]), run["detail"], {}
    for k in INDICATORS:
        v = flat.get(k)
        if isinstance(v, str):                                # e.g. "4.6%" from generic prompts
            m = re.search(r"-?\d+(?:\.\d+)?", v)
            v = float(m.group(0)) if m else None
            reasons[k] = {"fallback": "coerced_from_string"} if v is not None else {"fallback": "unparseable_string"}
        lo, hi = INDICATOR_SPECS[k]["range"]
        if v is not None and not (lo <= v <= hi):
            reasons[k] = {"fallback": "out_of_plausible_range", "raw": v}
            v = None
        if v is None and k not in reasons:
            d = detail.get(k, {}) if detail else {}
            reasons[k] = {"fallback": d.get("status") or ("extraction_failed" if run["error"] else "not_reported"),
                          "model_note": d.get("note"), "proxy_value": d.get("proxy_value"),
                          "proxy_definition": d.get("proxy_definition")}
        flat[k] = v
    flat["country_name"] = flat.get("country_name") or C
    return CoreIndicators(**flat), reasons


MAIN_RUN = run_indicator_extraction(EXTRACTOR_MODEL, "strict")
CORE, FALLBACKS = apply_fallbacks(MAIN_RUN)
print("Schema compliance:", MAIN_RUN["compliance"])
display(pd.DataFrame([{"indicator": k, "value": getattr(CORE, k),
                       "year": MAIN_RUN["detail"].get(k, {}).get("reference_year") if MAIN_RUN["detail"] else None,
                       "status": MAIN_RUN["detail"].get(k, {}).get("status") if MAIN_RUN["detail"] else None,
                       "source": MAIN_RUN["detail"].get(k, {}).get("source_chunk") if MAIN_RUN["detail"] else None,
                       "quote": (MAIN_RUN["detail"].get(k, {}).get("source_quote") or "")[:90] if MAIN_RUN["detail"] else "",
                       "fallback": FALLBACKS.get(k, {}).get("fallback")} for k in INDICATORS]))

### 2.4 Evaluator validation of the JSON: judge audit + human ground truth
Two independent checks:
* **LLM-as-a-judge audit.** The judge sees each extracted value, its quote and the *same* retrieved context, and labels it `supported`, `unsupported`, `contradicted`, `correct_null` or `missed_value`.
* **Human ground truth** (`ground_truth.json`). Values I verified by hand against the PDF pages, with accepted alternatives where the definition is genuinely ambiguous (end-of-period vs period-average inflation, for example). This measures **accuracy**, and also measures how far the judge itself can be trusted: where the judge and the ground truth disagree, the judge's error rate shows.

In [ ]:
INDICATOR_AUDIT_PROMPT = """Audit these indicator values extracted for {country} against the CONTEXT.
For each indicator choose a verdict:
- supported: the value matches the context and the indicator definition;
- unsupported: the value cannot be found in the context;
- contradicted: the context gives a different value or the value belongs to another series/year/institution type;
- correct_null: the value is null and the context indeed does not report this indicator;
- missed_value: the value is null but the context does report it (give expected_value).
Use the definitions strictly.

Definitions:
{definitions}

EXTRACTED (JSON):
{extracted}

CONTEXT:
{context}"""
PROMPTS["INDICATOR_AUDIT_PROMPT"] = INDICATOR_AUDIT_PROMPT


class IndicatorVerdict(BaseModel):
    indicator: str
    verdict: str
    expected_value: Optional[float] = None
    comment: str = ""


class IndicatorAudit(BaseModel):
    verdicts: List[IndicatorVerdict]
    overall_score: int = Field(description="1-5 overall factual accuracy")


def audit_indicators(run: Dict[str, Any], tag: str) -> Dict[str, Any]:
    extracted = run["detail"] or {k: {"value": run["flat"].get(k)} for k in INDICATORS}
    extracted = {k: {f: v for f, v in extracted.get(k, {}).items() if f in ("value", "reference_year", "source_quote",
                                                                          "source_chunk", "proxy_value")}
                 for k in INDICATORS}
    res = llm.generate(INDICATOR_AUDIT_PROMPT.format(country=C, definitions=DEFINITIONS,
                                                     extracted=json.dumps(extracted, indent=1), context=INDICATOR_CONTEXT),
                       JUDGE_MODEL, system=SYSTEM_JUDGE, schema=IndicatorAudit, tag=tag)
    if not res.ok:
        return {"error": res.error, "verdicts": {}, "support_rate": np.nan, "overall_score": np.nan}
    verdicts = {v.indicator: v.model_dump() for v in res.parsed.verdicts}
    good = sum(v["verdict"] in ("supported", "correct_null") for v in verdicts.values())
    return {"verdicts": verdicts, "support_rate": good / len(INDICATORS),
            "overall_score": int(min(5, max(1, res.parsed.overall_score)))}


GT_PATH = BASE_DIR / "ground_truth.json"
GROUND_TRUTH = json.loads(GT_PATH.read_text())["indicators"] if GT_PATH.exists() else {}
if not GROUND_TRUTH:
    print("⚠️  ground_truth.json not found – accuracy metrics will be NaN.")


def accuracy_vs_truth(flat: Dict[str, Any], tol: float = 0.051) -> Dict[str, Any]:
    per = {}
    for k in INDICATORS:
        gt = GROUND_TRUTH.get(k)
        if gt is None:
            continue
        accepted = [gt["value"]] + gt.get("accepted_alternatives", [])
        v = flat.get(k)
        if gt["value"] is None:
            per[k] = v is None
        else:
            per[k] = isinstance(v, (int, float)) and any(a is not None and abs(v - a) <= tol for a in accepted)
    return {"per_indicator": per, "accuracy": (sum(per.values()) / len(per)) if per else np.nan}


MAIN_AUDIT = audit_indicators(MAIN_RUN, "indicator-audit-main")
MAIN_ACC = accuracy_vs_truth(CORE.model_dump())
audit_table = pd.DataFrame([{
    "indicator": k, "extracted": getattr(CORE, k),
    "ground_truth": GROUND_TRUTH.get(k, {}).get("value"), "correct": MAIN_ACC["per_indicator"].get(k),
    "judge_verdict": MAIN_AUDIT["verdicts"].get(k, {}).get("verdict"),
    "judge_agrees_with_truth": (MAIN_AUDIT["verdicts"].get(k, {}).get("verdict") in ("supported", "correct_null"))
                               == MAIN_ACC["per_indicator"].get(k) if k in MAIN_ACC["per_indicator"] else None,
    "fallback": FALLBACKS.get(k, {}).get("fallback")} for k in INDICATORS])
display(audit_table)
print(f"Accuracy vs human ground truth: {MAIN_ACC['accuracy']:.0%} | judge support rate: {MAIN_AUDIT['support_rate']:.0%}")

save_json({"country": C, "report": CONFIG["imf_country_report_no"], "model": EXTRACTOR_MODEL,
           "indicators": CORE.model_dump(), "fallback_value": None, "fallbacks": FALLBACKS,
           "evidence": MAIN_RUN["detail"], "schema_compliance": MAIN_RUN["compliance"],
           "context_chunks": INDICATOR_CHUNKS}, "core_indicators.json")
save_json({"judge_model": JUDGE_MODEL, "audit": MAIN_AUDIT, "accuracy_vs_ground_truth": MAIN_ACC,
           "table": audit_table.to_dict(orient="records")}, "indicator_audit.json")

### 2.5 Temporal trends for longitudinal plotting
* **Deterministic table parser** (the reference). It walks the layout-mode lines, tracks the current table title, year header and row group (e.g. *Consumer prices → Period average*), and keeps a row only when its count of numbers **exactly equals** the count of year columns. Ragged rows are skipped and counted, never guessed. Years ≥ `projection_start_year` are flagged as projections.
* **LLM trend identification.** The extractor scans the retrieved table chunks and proposes the series most relevant to financial stability. Every LLM value is then **cross-checked against the parser**, which gives an agreement rate. The dashboard plots the parser's values, so every number on the chart traces to a table cell.

In [ ]:
def parse_number(tok: str) -> float:
    if MISSING_TOKEN.match(tok):
        return np.nan
    neg = tok.startswith("(") and tok.endswith(")")
    v = float(tok.strip("()%").replace(",", ""))
    return -v if neg else v


def split_row(line: str) -> Tuple[str, List[float]]:
    toks = line.split()
    nums = []
    while toks and (NUM_TOKEN.match(toks[-1]) or MISSING_TOKEN.match(toks[-1])):
        nums.insert(0, parse_number(toks.pop()))
    label = re.sub(r"\s*\b\d{1,2}/", "", " ".join(toks)).strip(" :")
    return label, nums


def parse_tables(pages: List[Dict[str, Any]]) -> Tuple[pd.DataFrame, int]:
    rows, skipped = [], 0
    title, years, group = None, None, None
    for pg in pages:
        if not page_is_tabular(pg["lines"]) and not any(TITLE_LINE.match(l) for l in pg["lines"]):
            title, years, group = None, None, None          # a prose page ends any open table
        for raw in pg["layout_lines"]:
            s = re.sub(r"\s+", " ", raw).strip()
            if TITLE_LINE.match(s):
                title, years, group = (s if re.search(r"\bTable\b", s) else None), None, None
                continue
            if title is None:
                continue
            ys = years_in(s)
            numeric = [t for t in s.split() if NUM_TOKEN.match(t)]
            if len(ys) >= 4 and len(ys) >= 0.8 * len(numeric):
                years = ys
                continue
            label, nums = split_row(s)
            if not nums:
                if label and len(label) < 90 and not label.lower().startswith(("source", "note")):
                    group = label
                continue
            if years is None or not label or len(nums) < 3:
                continue                                    # a trailing year/number in prose is not a row
            if len(nums) != len(years):
                skipped += 1
                continue
            for y, v in zip(years, nums):
                if not np.isnan(v):
                    rows.append({"table": title, "group": group, "series": label, "year": y, "value": v,
                                 "page": pg["page"], "is_projection": y >= CONFIG["projection_start_year"]})
    df = pd.DataFrame(rows)
    if not df.empty:
        df["table_key"] = df["table"].str.replace(r"\s*\(continued\)|\s*\(concluded\)", "", regex=True)
        df = df.drop_duplicates(subset=["table_key", "group", "series", "year"], keep="first")
    return df, skipped


TRENDS_DF, n_skipped = parse_tables(PAGES)
print(f"Parsed {len(TRENDS_DF):,} table cells from {TRENDS_DF['table_key'].nunique() if not TRENDS_DF.empty else 0} "
      f"tables; skipped {n_skipped} ragged rows")


def find_series(table_rx: str, series_rx: str, group_rx: Optional[str] = None) -> pd.Series:
    """Year-indexed values of one parsed series (first matching table/series), empty if not found."""
    if TRENDS_DF.empty:
        return pd.Series(dtype=float)
    m = TRENDS_DF["table_key"].str.contains(table_rx, case=False, regex=True) & \
        TRENDS_DF["series"].str.contains(series_rx, case=False, regex=True)
    if group_rx:
        m &= TRENDS_DF["group"].fillna("").str.contains(group_rx, case=False, regex=True)
    sub = TRENDS_DF[m]
    if sub.empty:
        return pd.Series(dtype=float)
    first = sub.groupby(["table_key", "series"], sort=False).ngroup() == 0
    return sub[first].set_index("year")["value"].sort_index()


KEY_SERIES = {
    "Real GDP growth (%)": find_series(r"Selected Economic Indicators", r"^Real GDP"),
    "CPI inflation, period average (%)": find_series(r"Selected Economic Indicators", r"^Period average", r"Consumer prices"),
    "CPI inflation, end of period (%)": find_series(r"Selected Economic Indicators", r"^End of period", r"Consumer prices"),
    "Credit to private sector growth (%)": find_series(r"Selected Economic Indicators", r"Credit to the private sector"),
    "Central government debt (% of GDP)": find_series(r"Selected Economic Indicators", r"^Central government debt"),
    "Bank NPL ratio (%)": find_series(r"Commercial Banks", r"^Nonperforming loans to total gross loans"),
    "Bank regulatory capital / RWA (%)": find_series(r"Commercial Banks", r"^Regulatory capital ratio"),
    "Credit union NPL ratio (%)": find_series(r"Credit Unions", r"^Nonperforming loans to total gross loans"),
    "Credit union regulatory capital (%)": find_series(r"Credit Unions", r"^Regulatory capital ratio"),
}
KEY_WIDE = pd.DataFrame(KEY_SERIES).T                       # wide: series × year
KEY_WIDE.columns = KEY_WIDE.columns.astype(int)
display(KEY_WIDE.round(2))

In [ ]:
TREND_PROMPT = """The CONTEXT contains statistical tables from the IMF Article IV report on {country}.
Identify up to 8 time series most relevant to macro-financial stability trends (e.g. growth, inflation, credit, public debt,
bank and credit-union NPLs and capital). For each: name (row label as written), unit, source_table (table title as written),
theme (one of: {themes}; or "Growth & Outlook"), and points = every year/value pair shown in that row.
Copy numbers exactly; never interpolate. Use the row under the correct table title.

CONTEXT:
{context}"""
PROMPTS["TREND_PROMPT"] = TREND_PROMPT


class TrendPoint(BaseModel):
    year: int
    value: float


class TrendSeries(BaseModel):
    name: str
    unit: str
    source_table: str
    theme: str
    points: List[TrendPoint]


class TrendSet(BaseModel):
    series: List[TrendSeries]


trend_hits = []
for q in ["Selected Economic Indicators output prices money and credit", "Financial Stability Indicators commercial banks",
          "Financial Stability Indicators credit unions", "central government debt percent of GDP projected"]:
    trend_hits += VINDEX.search(q, k=3, kind="table")
TREND_CONTEXT = format_context(trend_hits, max_tokens=8000)
trend_res = llm.generate(TREND_PROMPT.format(country=C, themes="; ".join(THEMES), context=TREND_CONTEXT),
                         EXTRACTOR_MODEL, system=SYSTEM_ANALYST, schema=TrendSet, tag="trends")
LLM_TRENDS = trend_res.parsed.series if trend_res.ok else []


def _tok(s: str) -> set:
    return set(re.findall(r"[a-z]+", s.lower())) - {"of", "the", "to", "in", "and", "percent", "total"}


def cross_check_trend(ts: TrendSeries) -> Dict[str, Any]:
    """Match an LLM series to the parsed table row with the highest name overlap (table title as tie-breaker)."""
    if TRENDS_DF.empty:
        return {"matched_series": None, "agreement": np.nan, "n_compared": 0}
    cands = TRENDS_DF[["table_key", "group", "series"]].drop_duplicates()
    def score(r):
        a, b = _tok(ts.name), _tok(r.series) | _tok(r.group or "")
        jac = len(a & b) / max(1, len(a | b))
        return jac + 0.25 * (len(_tok(ts.source_table) & _tok(r.table_key)) / max(1, len(_tok(ts.source_table))))
    best = max(cands.itertuples(), key=score)
    ref = TRENDS_DF[(TRENDS_DF.table_key == best.table_key) & (TRENDS_DF.series == best.series) &
                    (TRENDS_DF.group.fillna("") == (best.group or ""))].set_index("year")["value"]
    pairs = [(p.value, ref[p.year]) for p in ts.points if p.year in ref.index]
    agree = sum(abs(a - b) <= 0.051 for a, b in pairs)
    return {"matched_table": best.table_key, "matched_series": best.series, "n_compared": len(pairs),
            "agreement": agree / len(pairs) if pairs else np.nan}


TREND_CHECK = pd.DataFrame([{"llm_series": t.name, "llm_table": t.source_table, "theme": t.theme,
                             "n_points": len(t.points), **cross_check_trend(t)} for t in LLM_TRENDS])
display(TREND_CHECK)
if not TREND_CHECK.empty:
    print(f"LLM vs parser value agreement (mean over series): {TREND_CHECK['agreement'].mean():.0%}")
save_json({"parser": {"cells": len(TRENDS_DF), "skipped_ragged_rows": n_skipped,
                      "key_series": {k: {int(y): float(v) for y, v in s.items()} for k, s in KEY_SERIES.items()},
                      "projection_start_year": CONFIG["projection_start_year"]},
           "llm_series": [t.model_dump() for t in LLM_TRENDS],
           "cross_check": TREND_CHECK.to_dict(orient="records")}, "temporal_trends.json")
TRENDS_DF.to_csv(OUT_DIR / "parsed_table_cells.csv", index=False)

---
## Task 3 — RAG grounding and retrieval
The FAISS index from §1.4 now answers factual policy questions about the report. Each question runs under three conditions:
1. **Closed-book** (no retrieval): the model answers from its parametric memory. This is the *hallucination baseline*.
2. **Single-query RAG**: top-k = 5 chunks for the original question.
3. **Multi-Query RAG** (*extension b*): the LLM writes 3 paraphrases, each query retrieves top-k chunks, and the ranked lists are fused with **Reciprocal Rank Fusion** (Cormack et al., 2009) into a top-8 context.

Grounded answers must cite chunk ids, and the cited ids are checked against the context the model was actually given. The **judge scores all three answers against one shared evidence pool** (the union of both retrievals), so that faithfulness and completeness are comparable across conditions. A cheap **retrieval hit test** (do the expected key terms appear in the retrieved text?) checks retrieval quality separately from generation quality.

In [ ]:
RAG_QUESTIONS = [
    {"q": "What does the IMF recommend on bank loan-loss provisioning and capital for credit unions?",
     "expect": ["provision", "credit union", "capital"]},
    {"q": "Which shocks and assumptions are used in the debt sustainability stress tests, such as a natural disaster scenario?",
     "expect": ["stress test", "natural disaster"]},
    {"q": "What is the outlook for inflation and how quickly is it expected to converge to about 2 percent?",
     "expect": ["inflation", "2 percent"]},
    {"q": "How much do fiscal accounts rely on Citizenship by Investment revenues and what risk does that create?",
     "expect": ["CBI", "revenue"]},
    {"q": "What public debt target applies to Dominica and by when is it expected to be reached?",
     "expect": ["60 percent", "debt"]},
]

RAG_ANSWER_PROMPT = """Answer the QUESTION about {country} using ONLY the numbered CONTEXT passages.
- Cite the supporting passage ids in square brackets after each claim, e.g. [C012].
- If the context does not contain the answer, reply exactly: "Not stated in the retrieved context."
- Max 120 words. Keep figures and years exactly as written.

QUESTION: {question}

CONTEXT:
{context}"""
CLOSED_BOOK_PROMPT = """Answer the question about the IMF's 2024 Article IV consultation with {country} from your own knowledge (max 120 words).
QUESTION: {question}"""
MULTI_QUERY_PROMPT = """Generate {n} alternative search queries that would retrieve passages answering the question below from an
IMF Article IV report on {country}. Vary vocabulary (technical synonyms, related indicators) while keeping the same intent.
QUESTION: {question}"""
RAG_JUDGE_PROMPT = """Evaluate an ANSWER to a QUESTION about {country} against the EVIDENCE passages from the IMF report.
Score integers 1-5:
- faithfulness: 5 = every claim is supported by the evidence; 3 = one unsupported claim; 1 = mostly unsupported or fabricated.
- completeness: 5 = covers everything the evidence says that answers the question; 3 = misses an important element; 1 = misses most.
- relevance: 5 = directly answers the question; 1 = off-topic.
List unsupported claims verbatim.

QUESTION: {question}
ANSWER: {answer}
EVIDENCE:
{evidence}"""
PROMPTS.update({"RAG_ANSWER_PROMPT": RAG_ANSWER_PROMPT, "CLOSED_BOOK_PROMPT": CLOSED_BOOK_PROMPT,
                "MULTI_QUERY_PROMPT": MULTI_QUERY_PROMPT, "RAG_JUDGE_PROMPT": RAG_JUDGE_PROMPT})


class QueryVariants(BaseModel):
    queries: List[str]


class RAGJudgement(BaseModel):
    faithfulness: int
    completeness: int
    relevance: int
    unsupported_claims: List[str] = Field(default_factory=list)
    rationale: str = ""


def rrf_fuse(ranked_lists: List[List[Tuple[Chunk, float]]], k_rrf: int, top_n: int) -> List[Tuple[Chunk, float]]:
    """Reciprocal Rank Fusion: score(d) = Σ 1 / (k + rank_d)."""
    scores, by_id = defaultdict(float), {}
    for lst in ranked_lists:
        for rank, (c, _) in enumerate(lst, start=1):
            scores[c.chunk_id] += 1.0 / (k_rrf + rank)
            by_id[c.chunk_id] = c
    best = sorted(scores, key=scores.get, reverse=True)[:top_n]
    return [(by_id[i], scores[i]) for i in best]


def retrieval_hit(hits: List[Tuple[Chunk, float]], expect: List[str]) -> float:
    text = " ".join(c.text.lower() for c, _ in hits)
    return sum(e.lower() in text for e in expect) / len(expect)


def cited_ids(answer: str) -> List[str]:
    return re.findall(r"C\d{3}", answer)


def judge_rag(question: str, answer: str, evidence: str, tag: str) -> Optional[RAGJudgement]:
    res = llm.generate(RAG_JUDGE_PROMPT.format(country=C, question=question, answer=answer, evidence=evidence),
                       JUDGE_MODEL, system=SYSTEM_JUDGE, schema=RAGJudgement, tag=tag)
    if not res.ok:
        return None
    j = res.parsed
    for f_ in ("faithfulness", "completeness", "relevance"):
        setattr(j, f_, int(min(5, max(1, getattr(j, f_)))))
    return j


RAG_RESULTS = []
for qi, item in enumerate(RAG_QUESTIONS, start=1):
    q = item["q"]
    single = VINDEX.search(q, k=CONFIG["retrieval_k"])
    mq_res = llm.generate(MULTI_QUERY_PROMPT.format(n=CONFIG["multi_query_n_variants"], country=C, question=q),
                          EXTRACTOR_MODEL, schema=QueryVariants, tag=f"rag{qi}-variants")
    variants = (mq_res.parsed.queries[:CONFIG["multi_query_n_variants"]] if mq_res.ok else [])
    multi = rrf_fuse([VINDEX.search(v, k=CONFIG["retrieval_k"]) for v in [q] + variants],
                     CONFIG["rrf_k"], CONFIG["multi_query_top_k"])
    evidence_pool = format_context(single + multi, max_tokens=12_000)

    conditions = {
        "closed_book": (None, llm.generate(CLOSED_BOOK_PROMPT.format(country=C, question=q), EXTRACTOR_MODEL,
                                           tag=f"rag{qi}-closed")),
        "single_query": (single, llm.generate(RAG_ANSWER_PROMPT.format(country=C, question=q, context=format_context(single)),
                                              EXTRACTOR_MODEL, system=SYSTEM_ANALYST, tag=f"rag{qi}-single")),
        "multi_query": (multi, llm.generate(RAG_ANSWER_PROMPT.format(country=C, question=q, context=format_context(multi)),
                                            EXTRACTOR_MODEL, system=SYSTEM_ANALYST, tag=f"rag{qi}-multi")),
    }
    rec = {"question": q, "variants": variants, "retrieval": {}, "answers": {}}
    for name, (hits, res) in conditions.items():
        answer = res.text.strip()
        given = {c.chunk_id for c, _ in hits} if hits else set()
        cites = cited_ids(answer)
        j = judge_rag(q, answer, evidence_pool, f"rag{qi}-judge-{name}")
        rec["answers"][name] = {
            "answer": answer, "latency_s": round(res.latency_s, 2), "error": res.error,
            "citations": cites, "citation_validity": (sum(c in given for c in cites) / len(cites)) if cites else None,
            "judge": j.model_dump() if j else None}
        if hits:
            rec["retrieval"][name] = {"chunks": [{"chunk_id": c.chunk_id, "section": c.section, "pages":
                                                  f"{c.page_start}-{c.page_end}", "score": round(s, 4)} for c, s in hits],
                                      "hit_rate": retrieval_hit(hits, item["expect"]),
                                      "unique_sections": len({c.sid for c, _ in hits})}
    RAG_RESULTS.append(rec)
    print(f"Q{qi}: hit single={rec['retrieval']['single_query']['hit_rate']:.2f} "
          f"multi={rec['retrieval']['multi_query']['hit_rate']:.2f}")

rag_rows = []
for qi, r in enumerate(RAG_RESULTS, start=1):
    for cond, a in r["answers"].items():
        if a["judge"]:
            rag_rows.append({"q": f"Q{qi}", "condition": cond, **{k: a["judge"][k] for k in
                             ("faithfulness", "completeness", "relevance")},
                             "n_unsupported": len(a["judge"]["unsupported_claims"]),
                             "citation_validity": a["citation_validity"],
                             "hit_rate": r["retrieval"].get(cond, {}).get("hit_rate")})
RAG_DF = pd.DataFrame(rag_rows)
if not RAG_DF.empty:
    display(RAG_DF.groupby("condition")[["faithfulness", "completeness", "relevance", "n_unsupported",
                                         "citation_validity", "hit_rate"]].mean().round(2))
display(Markdown(f"**Example (Q1, single-query RAG):** {RAG_RESULTS[0]['answers']['single_query']['answer']}"))
save_json({"embedding_model": CONFIG["embedding_model"], "index": "faiss.IndexFlatL2 (unit-normalised vectors)",
           "k": CONFIG["retrieval_k"], "multi_query_top_k": CONFIG["multi_query_top_k"], "results": RAG_RESULTS},
          "rag_results.json")

---
## Extension (a) — Model and system-directive benchmarking
The *same* indicator-extraction task, on the *same* retrieved context, runs as a 2 × 2 design:

| | strict (`system_instruction` + `response_schema`) | generic (plain user prompt, no schema) |
|---|---|---|
| Flash | ✔ | ✔ |
| Flash-Lite | ✔ | ✔ |

Metrics: **schema compliance** (strict key and type check), **latency** (seconds, first non-cached call), **accuracy vs the human ground truth**, and the **judge support rate**. One fixed judge scores all four cells, so differences come from the extractor configuration and not from the judge.

In [ ]:
BENCH_MODELS = [m for m in dict.fromkeys([EXTRACTOR_MODEL, LITE_MODEL]) if m]
BENCH_ROWS, BENCH_RUNS = [], {}
for model in BENCH_MODELS:
    for style in ("strict", "generic"):
        run = MAIN_RUN if (model == EXTRACTOR_MODEL and style == "strict") else run_indicator_extraction(model, style)
        core, fb = apply_fallbacks(run)
        audit = MAIN_AUDIT if run is MAIN_RUN else audit_indicators(run, f"indicator-audit-{style}-{model}")
        acc = accuracy_vs_truth(core.model_dump())
        BENCH_RUNS[f"{model}|{style}"] = {"run": run, "core": core.model_dump(), "audit": audit, "accuracy": acc}
        BENCH_ROWS.append({"model": model, "style": style, "label": f"{model.replace('gemini-', '')}\n{style}",
                           "schema_compliant": int(run["compliance"]["compliant"]),
                           "fields_populated": sum(core.model_dump()[k] is not None for k in INDICATORS),
                           "accuracy": acc["accuracy"], "judge_support_rate": audit["support_rate"],
                           "judge_overall_1to5": audit["overall_score"], "latency_s": run["latency_s"],
                           "error": run["error"]})
BENCH_DF = pd.DataFrame(BENCH_ROWS)
display(BENCH_DF.drop(columns=["label"]))
save_json({"design": "models x prompt style, same retrieved context, fixed judge", "judge_model": JUDGE_MODEL,
           "results": BENCH_DF.drop(columns=["label"]).to_dict(orient="records"),
           "runs": {k: {"core": v["core"], "audit": v["audit"], "accuracy": v["accuracy"],
                        "compliance": v["run"]["compliance"]} for k, v in BENCH_RUNS.items()}}, "benchmark.json")

---
## Task 4 — Tidy transformation and the multi-plot dashboard
### 4.1 Wide → long with `pd.melt`
Plotting libraries and grouped statistics both work best on **long (tidy)** data: one row per observation, with columns `series | year | value`. Every wide matrix the pipeline produced is melted here: indicator × year, core indicators, model benchmark × metric, judge scores × criterion, and theme × measure.

In [ ]:
# (1) Time-series matrix: series × year  →  long
KEY_LONG = (pd.melt(KEY_WIDE.reset_index(names="series"), id_vars="series", var_name="year", value_name="value")
            .dropna(subset=["value"]).astype({"year": int}))
KEY_LONG["is_projection"] = KEY_LONG["year"] >= CONFIG["projection_start_year"]

# (2) Core indicator JSON (one wide row) → long, nulls kept so fallbacks stay visible
CORE_LONG = pd.melt(pd.DataFrame([CORE.model_dump()]), id_vars="country_name", var_name="indicator", value_name="value")
CORE_LONG["fallback"] = CORE_LONG["indicator"].map(lambda k: FALLBACKS.get(k, {}).get("fallback"))

# (3) Model benchmark: metrics in columns → long
BENCH_LONG = pd.melt(BENCH_DF, id_vars=["model", "style", "label", "latency_s"],
                     value_vars=["accuracy", "schema_compliant", "judge_support_rate"],
                     var_name="metric", value_name="score")
BENCH_LONG["score_pct"] = 100 * BENCH_LONG["score"].astype(float)

# (4) Judge scores: criteria in columns → long (section summaries + RAG answers)
SUM_LONG = pd.melt(SUMMARY_JUDGE_DF, id_vars=["id", "round"],
                   value_vars=["consistency", "completeness", "factual_alignment"],
                   var_name="criterion", value_name="score") if not SUMMARY_JUDGE_DF.empty else pd.DataFrame()
RAG_LONG = pd.melt(RAG_DF, id_vars=["q", "condition"], value_vars=["faithfulness", "completeness", "relevance"],
                   var_name="criterion", value_name="score") if not RAG_DF.empty else pd.DataFrame()

# (5) Themes: two measures → long
THEME_LONG = pd.melt(THEME_DF, id_vars="theme", value_vars=["lexicon_share_pct", "embedding_share_pct"],
                     var_name="measure", value_name="share_pct")

for name, df in [("KEY_LONG", KEY_LONG), ("CORE_LONG", CORE_LONG), ("BENCH_LONG", BENCH_LONG),
                 ("SUM_LONG", SUM_LONG), ("RAG_LONG", RAG_LONG), ("THEME_LONG", THEME_LONG)]:
    print(f"{name:<11} {df.shape}")
display(KEY_LONG.head(8))
KEY_LONG.to_csv(OUT_DIR / "key_series_long.csv", index=False)

### 4.2 Dashboard
Six panels, each answering one question:

| Panel | Question | Form |
|---|---|---|
| A | Where does the report put its attention? | horizontal bars, lexicon vs embedding share |
| B | How do growth, inflation and credit move over 2019–29? | lines, with projections shaded |
| C | Does weak credit growth coincide with high NPLs? | scatter, banks vs credit unions, labelled by year |
| D | Are bank and credit-union balance sheets getting healthier? | lines, 2013–23 |
| E | Which model and prompt set-up extracts best? | grouped bars (%), latency annotated |
| F | How much can the generated text be trusted? | box plot of judge scores by output type |

Design rules: one y-axis per panel (no dual axes), every value in **percent units** (so no 0.04-vs-4% scale errors), a fixed colour-blind-safe categorical order, grid lines, units in the axis labels, and an executive text band at the bottom. Saved at **300 dpi**, together with every panel as its own file for the report.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]   # fixed order
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
plt.rcParams.update({"font.size": 12, "axes.titlesize": 15, "axes.titleweight": "bold", "axes.labelsize": 12,
                     "legend.fontsize": 10.5, "xtick.labelsize": 11, "ytick.labelsize": 11,
                     "axes.edgecolor": INK2, "axes.labelcolor": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": "white"})


def style_ax(ax, title: str, xlabel: str = "", ylabel: str = ""):
    ax.set_title(title, loc="left", pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, color=GRID, linewidth=0.8, alpha=0.8)
    ax.set_axisbelow(True)


def legend_below(ax, ncol: int = 3):
    """Legends sit under the plot area so they never cover data."""
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=ncol, frameon=False)


def no_data(ax, msg: str):
    ax.text(0.5, 0.5, msg, ha="center", va="center", color=INK2, fontsize=12, transform=ax.transAxes, wrap=True)


def panel_themes(ax):
    d = THEME_DF.sort_values("lexicon_share_pct")
    y = np.arange(len(d))
    ax.barh(y + 0.2, d["lexicon_share_pct"], height=0.38, color=PALETTE[0], label="Lexicon mentions (share %)")
    ax.barh(y - 0.2, d["embedding_share_pct"], height=0.38, color=PALETTE[1], label="Embedding-classified chunks (share %)")
    for yi, v, n in zip(y, d["lexicon_share_pct"], d["mentions_per_1k_words"]):
        ax.text(v + 0.5, yi + 0.2, f"{n:.1f}/1k words", va="center", fontsize=9.5, color=INK2)
    ax.set_yticks(y, d["theme"])
    ax.set_xlim(0, max(d[["lexicon_share_pct", "embedding_share_pct"]].max()) * 1.3 + 1)
    style_ax(ax, "A. Thematic density across the 7 financial-stability themes", "Share of total (%)")
    ax.legend(loc="lower right", frameon=False)


def panel_macro(ax):
    names = ["Real GDP growth (%)", "CPI inflation, period average (%)", "Credit to private sector growth (%)"]
    plotted = False
    for i, n in enumerate(names):
        s = KEY_SERIES.get(n, pd.Series(dtype=float))
        if s.empty:
            continue
        s = s[(s.index >= 2015)]
        ax.plot(s.index, s.values, color=PALETTE[i], lw=2.2, marker="o", ms=5, label=n)
        ax.annotate(f"{s.iloc[-1]:.1f}", (s.index[-1], s.iloc[-1]), xytext=(6, 0), textcoords="offset points",
                    va="center", fontsize=9.5, color=INK2)
        plotted = True
    if not plotted:
        return no_data(ax, "Selected Economic Indicators table not parsed")
    ax.axvspan(CONFIG["projection_start_year"] - 0.5, ax.get_xlim()[1], color="#f1f0ec", zorder=0)
    ax.text(CONFIG["projection_start_year"] - 0.3, 0.97, "IMF staff projections →", color=INK2, fontsize=10,
            va="top", transform=ax.get_xaxis_transform())
    ax.axhline(0, color=INK2, lw=0.8)
    style_ax(ax, "B. Growth, inflation and credit: history and projections",
             "Year   (no policy-rate series: Dominica uses the ECCB's quasi-currency board)", "Annual % change")
    legend_below(ax, ncol=3)


def panel_credit_npl(ax):
    credit = KEY_SERIES.get("Credit to private sector growth (%)", pd.Series(dtype=float))
    pairs = [("Commercial banks", "Bank NPL ratio (%)", PALETTE[0], "o"),
             ("Credit unions", "Credit union NPL ratio (%)", PALETTE[1], "s")]
    plotted = False
    for label, key, col, mk in pairs:
        npl = KEY_SERIES.get(key, pd.Series(dtype=float))
        yrs = sorted(set(credit.index) & set(npl.index))
        if not yrs:
            continue
        x, y = credit[yrs].values, npl[yrs].values
        ax.scatter(x, y, s=90, color=col, marker=mk, edgecolor="white", linewidth=1.5, label=label, zorder=3)
        for xi, yi, yr in zip(x, y, yrs):
            ax.annotate(str(yr), (xi, yi), xytext=(5, 5), textcoords="offset points", fontsize=9.5, color=INK2)
        plotted = True
    if not plotted:
        return no_data(ax, "No overlapping years between credit growth and NPL tables")
    ax.axvline(0, color=INK2, lw=0.8)
    style_ax(ax, "C. Private credit growth vs NPL ratio (overlapping years)",
             "Credit to private sector, annual % change", "NPLs / gross loans (%)")
    legend_below(ax, ncol=2)


def panel_banking(ax):
    spec = [("Bank NPL ratio (%)", PALETTE[0], "-"), ("Credit union NPL ratio (%)", PALETTE[1], "-"),
            ("Bank regulatory capital / RWA (%)", PALETTE[0], "--"),
            ("Credit union regulatory capital (%)", PALETTE[1], "--")]
    plotted = False
    for name, col, ls in spec:
        s = KEY_SERIES.get(name, pd.Series(dtype=float))
        if s.empty:
            continue
        ax.plot(s.index, s.values, color=col, ls=ls, lw=2.2, marker="o", ms=4.5, label=name)
        ax.annotate(f"{s.iloc[-1]:.1f}", (s.index[-1], s.iloc[-1]), xytext=(6, 0), textcoords="offset points",
                    va="center", fontsize=9.5, color=INK2)
        plotted = True
    if not plotted:
        return no_data(ax, "Financial Stability Indicator tables not parsed")
    style_ax(ax, "D. Balance-sheet soundness: NPLs (solid) vs capital (dashed)", "Year", "Percent")
    legend_below(ax, ncol=2)


def panel_models(ax):
    if BENCH_DF.empty:
        return no_data(ax, "Benchmark not run")
    metrics = [("accuracy", "Accuracy vs human ground truth"), ("schema_compliant", "Schema compliance"),
               ("judge_support_rate", "Judge support rate")]
    labels = BENCH_DF["label"].tolist()
    x, w = np.arange(len(labels)), 0.26
    for i, (m, name) in enumerate(metrics):
        vals = 100 * BENCH_DF[m].astype(float).fillna(0).values
        ax.bar(x + (i - 1) * w, vals, width=w - 0.02, color=PALETTE[i], label=name)
    for xi, lat in zip(x, BENCH_DF["latency_s"]):
        ax.text(xi, 104, f"{lat:.1f}s", ha="center", fontsize=9.5, color=INK2)
    ax.set_xticks(x, labels)
    ax.set_ylim(0, 115)
    style_ax(ax, "E. Indicator extraction: model × prompt benchmark", "", "Score (%)  ·  latency shown above bars")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)


def panel_judge(ax):
    groups = []
    if not SUMMARY_JUDGE_DF.empty:
        crit = ["consistency", "completeness", "factual_alignment"]
        first = SUMMARY_JUDGE_DF[SUMMARY_JUDGE_DF["round"] == 0]
        final = SUMMARY_JUDGE_DF.sort_values("round").groupby("id").tail(1)
        groups += [("Summaries\n(draft)", first[crit].mean(axis=1)), ("Summaries\n(final)", final[crit].mean(axis=1))]
    if not RAG_DF.empty:
        crit = ["faithfulness", "completeness", "relevance"]
        for cond, lab in [("closed_book", "RAG: closed\nbook"), ("single_query", "RAG: single\nquery"),
                          ("multi_query", "RAG: multi\nquery")]:
            groups.append((lab, RAG_DF[RAG_DF.condition == cond][crit].mean(axis=1)))
    groups = [(l, v.dropna()) for l, v in groups if len(v.dropna())]
    if not groups:
        return no_data(ax, "No judge scores available")
    bp = ax.boxplot([v.values for _, v in groups], patch_artist=True, widths=0.5,
                    medianprops={"color": INK, "lw": 2})
    for k, patch in enumerate(bp["boxes"]):
        patch.set_facecolor(PALETTE[min(k, len(PALETTE) - 1)])
        patch.set_alpha(0.35)
    rng = np.random.default_rng(7)
    for k, (_, v) in enumerate(groups, start=1):
        ax.scatter(k + rng.uniform(-0.12, 0.12, len(v)), v.values, s=26, color=INK2, zorder=3)
    ax.set_xticks(range(1, len(groups) + 1), [l for l, _ in groups])
    ax.set_ylim(0.8, 5.3)
    ax.axhline(CONFIG["judge_pass_threshold"], color=PALETTE[6], ls=":", lw=1.5)
    ax.text(len(groups) + 0.45, CONFIG["judge_pass_threshold"] + 0.06, "pass threshold", color=PALETTE[6],
            fontsize=9.5, ha="right")
    style_ax(ax, f"F. LLM-as-a-judge scores ({JUDGE_MODEL})", "", "Mean criterion score (1-5)")


def executive_text() -> str:
    def fmt(k, unit="%"):
        v = getattr(CORE, k)
        return f"{v:.1f}{unit}" if v is not None else "n/a"
    fb = ", ".join(f"{k.replace('_pct', '').replace('_', ' ')} ({v['fallback']})" for k, v in FALLBACKS.items()) or "none"
    top_theme = THEME_DF.iloc[0]["theme"]
    vul = "; ".join(i["title"] for i in RISK_INSIGHTS["vulnerabilities"][:3]) or "n/a"
    acc = MAIN_ACC["accuracy"]
    return (f"{C} — {CONFIG['imf_country_report_no']}  |  GDP growth forecast {fmt('gdp_growth_forecast_pct')}  ·  "
            f"Headline inflation {fmt('headline_inflation_rate_pct')}  ·  Bank NPL ratio {fmt('non_performing_loan_ratio')}  ·  "
            f"Unemployment {fmt('unemployment_rate_pct')}  ·  Policy rate {fmt('central_bank_policy_rate_pct')}  ·  "
            f"Tier 1 {fmt('tier_1_capital_ratio')}\n"
            f"Most-discussed theme: {top_theme}.   Key vulnerabilities: {vul}.\n"
            f"Fallbacks (null): {fb}.   Extraction accuracy vs human ground truth: "
            f"{'n/a' if pd.isna(acc) else f'{acc:.0%}'}  ·  judge support rate {MAIN_AUDIT['support_rate']:.0%}.")


PANELS = [("A_themes", panel_themes), ("B_macro", panel_macro), ("C_credit_vs_npl", panel_credit_npl),
          ("D_banking", panel_banking), ("E_model_benchmark", panel_models), ("F_judge_scores", panel_judge)]

fig = plt.figure(figsize=(20, 23))
gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 0.22], hspace=0.55, wspace=0.22, top=0.95)
for k, (_, fn) in enumerate(PANELS):
    fn(fig.add_subplot(gs[k // 2, k % 2]))
txt_ax = fig.add_subplot(gs[3, :])
txt_ax.axis("off")
txt_ax.text(0.0, 0.95, "Executive overview", fontsize=14, fontweight="bold", va="top")
txt_ax.text(0.0, 0.62, executive_text(), fontsize=11.5, va="top", linespacing=1.6,
            bbox={"boxstyle": "round,pad=0.6", "facecolor": "#f6f5f2", "edgecolor": GRID})
fig.suptitle(f"{C}: IMF 2024 Article IV — LLM-extracted macro-financial dashboard", fontsize=20,
             fontweight="bold", x=0.06, ha="left", y=0.99)
fig.text(0.06, 0.967, f"Extractor: {EXTRACTOR_MODEL}  ·  Judge: {JUDGE_MODEL}  ·  Embeddings: {CONFIG['embedding_model']}"
         f"  ·  Source: {CONFIG['imf_country_report_no']}, tables parsed from the PDF", fontsize=11, color=INK2)
DASHBOARD_PATH = FIG_DIR / "dashboard.png"
fig.savefig(DASHBOARD_PATH, dpi=300, bbox_inches="tight")
plt.show()

for name, fn in PANELS:                                     # individual high-resolution panels for the report
    f, ax = plt.subplots(figsize=(10, 6.8))
    fn(ax)
    f.tight_layout()
    f.savefig(FIG_DIR / f"panel_{name}.png", dpi=300, bbox_inches="tight")
    plt.close(f)
print("Saved:", DASHBOARD_PATH, "and", len(PANELS), "individual panels (300 dpi)")

---
## 5. Run summary and artefact index
Every number below comes from the files in `outputs/`. These are the numbers the written report quotes.

In [ ]:
pd.DataFrame(llm.log).to_csv(OUT_DIR / "llm_call_log.csv", index=False)
save_json(PROMPTS, "prompts.json")
log_df = pd.DataFrame(llm.log)
summary = {
    "LLM calls (total / served from cache / failed)":
        f"{len(log_df)} / {int(log_df['cached'].sum()) if len(log_df) else 0} / {int((~log_df['ok']).sum()) if len(log_df) else 0}",
    "Pages / sections summarised / chunks": f"{len(PAGES)} / {len(SECTION_SUMMARIES)} / {len(CHUNKS)}",
    "Section summaries within 100 words": f"{sum(s['within_limit'] for s in SECTION_SUMMARIES)}/{len(SECTION_SUMMARIES)}",
    "Mean judge score, summaries (draft → final)":
        (f"{SUMMARY_JUDGE_DF[SUMMARY_JUDGE_DF['round'] == 0][['consistency', 'completeness', 'factual_alignment']].mean().mean():.2f} → "
         f"{SUMMARY_JUDGE_DF.sort_values('round').groupby('id').tail(1)[['consistency', 'completeness', 'factual_alignment']].mean().mean():.2f}")
        if not SUMMARY_JUDGE_DF.empty else "n/a",
    "Exec-summary quotes verified": f"{sum(b['quote_verified'] for b in EXEC_SUMMARY['bullets'])}/{len(EXEC_SUMMARY['bullets'])}",
    "Insight quotes verified": f"{sum(i['quote_verified'] for k in ('growth_drivers', 'vulnerabilities') for i in RISK_INSIGHTS[k])}"
                               f"/{len(RISK_INSIGHTS['growth_drivers']) + len(RISK_INSIGHTS['vulnerabilities'])}",
    "Indicator accuracy vs ground truth": "n/a" if pd.isna(MAIN_ACC["accuracy"]) else f"{MAIN_ACC['accuracy']:.0%}",
    "Indicator fallbacks": ", ".join(FALLBACKS) or "none",
    "Mean RAG faithfulness (closed / single / multi)":
        " / ".join(f"{RAG_DF[RAG_DF.condition == c]['faithfulness'].mean():.2f}" for c in
                   ("closed_book", "single_query", "multi_query")) if not RAG_DF.empty else "n/a",
}
display(pd.DataFrame(summary.items(), columns=["Metric", "Value"]))
print("\nArtefacts:")
for p in sorted(OUT_DIR.glob("*")) + sorted(FIG_DIR.glob("*.png")):
    print(f"  {p.relative_to(BASE_DIR)}  ({p.stat().st_size / 1024:.0f} KB)")

## 6. Human-oversight checklist (to complete before writing the report)
The LLM steps are audited automatically, but the brief makes **you** responsible for verified accuracy. Before quoting any number:
1. Open the PDF at the page cited in `core_indicators.json → evidence` for each indicator, and confirm `ground_truth.json` against it. **p.36 especially**: check that the bank and credit-union rows have not been swapped.
2. Read every judge `unsupported_claims` entry in `summaries.json` and `rag_results.json`. Decide whether the judge was right; judges produce false positives too.
3. Look at the rows where `judge_agrees_with_truth` is `False` in the audit table (§2.4). Those are cases of judge error or judge bias worth discussing.
4. Check the section table in §1.3 against the report's table of contents (pp. 6–7).
5. Make sure every figure you put in the report is in `figures/` at 300 dpi and is referenced in the text.